# Cell 00 - skfolio Masterclass 01: Foundations & ETF Data

**by The Chart Truth**

Begin the masterclass by building the data foundation for ETF portfolio research.

In this notebook, you will learn how to:

- load a public ETF teaching dataset without private credentials
- inspect ETF coverage, metadata, and liquidity before trusting the universe
- reshape daily prices into a clean date-by-symbol matrix
- convert prices into linear returns, the input skfolio expects for portfolio work
- use normalized prices and return distributions to build intuition before optimization
- finish with reusable objects such as `prices_wide`, `returns`, `etf_universe`, and `return_distribution_summary`

Across the full masterclass, those objects become the base for risk analysis, correlations, baseline portfolios, optimization, validation, and drawdown-aware portfolio research. Notebook 01 deliberately stops at the return matrix so the later portfolio work has a clean foundation.

**Educational note:** This notebook is for learning and research only. It is not financial, investment, tax, or legal advice, and it is not a recommendation to buy, sell, or hold any security.

**Independence note:** This project is independent educational material. It is not endorsed by, sponsored by, or affiliated with any broker, ETF issuer, index provider, exchange, data provider, or skfolio project maintainer.

Historical data can help us practice a process. It cannot guarantee future results.


## Cell 01 - Masterclass Map And Build Path

This masterclass turns raw ETF price data into a complete portfolio research workflow. Use Colab's table of contents on the left to jump between sections, but remember that code cells still depend on earlier setup, imports, and data-loading cells.

Notebook 01 is the foundation notebook. It gets the environment ready, explains the core skfolio workflow, loads the public ETF dataset, checks the universe, and creates the return matrix that later notebooks will use.

### Beginner path

- **Setup and orientation:** install packages, import tools, and set the chart style.
- **Data foundation:** load the ETF dataset, validate the schema, and understand the available universe.
- **Returns foundation:** convert prices to returns and learn why raw prices are not enough for portfolio modeling.

### Intermediate path

- **Notebook 02:** create portfolio objects, compare risk, inspect correlations, and build simple baselines.
- **Notebook 03:** split the timeline into training and testing periods, then build the first optimized portfolio.

### Advanced path

- **Later notebooks:** compare model families, efficient frontiers, robust inputs, constraints, stress tests, and drawdown-aware rankings.
- **Roadmap topics:** log-return comparisons, alternative transformations, transaction costs, turnover, fractional-share implementation, live-trading readiness, benchmark tracking, and deeper skfolio validation workflows.

The goal is not to find a magic portfolio. The goal is to learn a repeatable process for turning market data into portfolio research.


## Cell 02 - How To Use This Notebook

Run this notebook from top to bottom the first time. Each cell builds on the cells above it, so skipping setup or data-loading cells can make later cells fail.

**Expected setup:** Python 3 CPU runtime. No GPU, broker login, Hugging Face account, or private API key is needed for the default path.

**Expected time:** about 15 to 25 minutes for a careful first run, depending on package install speed and how long you spend reading the outputs.

In Colab, the runtime is the temporary Python session that runs your code. If the runtime disconnects or restarts, the notebook file is still saved in Google Drive, but installed packages, imported libraries, and variables in memory are cleared.

If that happens, use this recovery pattern:

1. Reconnect the runtime.
2. Rerun Cell 04 through Cell 06.
3. Rerun Cell 14 through the last data cell you need.
4. Continue from the section you were working on.

Common issues are usually simple: a package import failed because Cell 04 was skipped, a variable is missing because the runtime restarted, or a dataset file is missing because Cell 15 needs to be rerun.

When a code cell creates a table or chart, pause for a moment before moving on. The notebook is designed to teach the reasoning process, not just produce final numbers.


## Cell 03 - Install The Workshop Packages

Before we load data or build portfolios, Colab needs the Python packages used in this notebook.

The next code cell installs the tools for:

- working with tables and time series
- reading the public ETF dataset
- making interactive charts
- building and validating portfolios with skfolio

**Runtime note:** package installs affect only the current Colab runtime. If the runtime restarts later, rerun the install cell before importing packages again.

**What to check after it runs:** the package explanation table should appear first, then a short setup completion message. Quiet install logs are normal, but warnings should not be the main thing you see.


In [ ]:
# Cell 04 - Install Packages

from IPython.display import Markdown, display

package_notes = [
    ("skfolio", "Portfolio optimization, portfolio objects, risk measures, and validation tools."),
    ("pandas", "Tables, time series, date indexes, joins, summaries, and data cleaning."),
    ("numpy", "Fast numerical arrays and calculations used underneath the analysis."),
    ("plotly", "Interactive charts for prices, returns, risk, correlations, and weights."),
    ("huggingface_hub", "Downloads the public ETF teaching dataset without private credentials."),
    ("pyarrow", "Reads Parquet files, the columnar data format used by the dataset."),
    ("scikit-learn", "Model-selection helpers and validation patterns used by skfolio workflows."),
    ("scipy", "Scientific optimization and statistics routines used by portfolio tooling."),
]

package_table = "\n".join(
    ["| Package | Why we use it |", "|---|---|"]
    + [f"| `{name}` | {purpose} |" for name, purpose in package_notes]
)

display(Markdown("### Cell 04 - Packages Used In This Workshop\n\n" + package_table))

%pip install -q skfolio pandas numpy plotly huggingface_hub pyarrow scikit-learn scipy

print("Cell 04 complete: package installation finished.")
print("If an import fails later, restart the runtime, then rerun setup cells from Cell 04.")


## Cell 05 - Imports And Versions

This cell loads the Python libraries that the rest of the workshop depends on. Before you run the code cell below, here is what it will do:

- import the main analysis tools: pandas, NumPy, Plotly, scipy, scikit-learn, skfolio, and Hugging Face Hub helpers
- import the skfolio classes and measures we will use later for portfolio research
- create a `versions` table so the runtime environment is easy to check
- confirm that setup from Cell 04 worked before we move into chart helpers and data loading

**What to check after it runs:** you should see a compact version table and a short completion message. If an import fails, rerun Cell 04, then rerun this cell.


In [ ]:
# Cell 05 - Imports And Versions

import importlib.metadata as importlib_metadata
import platform

from IPython.display import Markdown, display
import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
import scipy
import sklearn

import skfolio as skf
from huggingface_hub import hf_hub_download
from skfolio import Population, Portfolio
from skfolio.measures import ExtraRiskMeasure, PerfMeasure, RatioMeasure, RiskMeasure
from skfolio.optimization import MeanRisk, ObjectiveFunction
from skfolio.preprocessing import prices_to_returns


def package_version(distribution_name):
    """Return a package version string that is safe to display in Colab."""
    try:
        return importlib_metadata.version(distribution_name)
    except importlib_metadata.PackageNotFoundError:
        return "not found"


version_rows = [
    ("Python", platform.python_version()),
    ("skfolio", package_version("skfolio")),
    ("pandas", pd.__version__),
    ("numpy", np.__version__),
    ("plotly", plotly.__version__),
    ("huggingface_hub", package_version("huggingface_hub")),
    ("pyarrow", package_version("pyarrow")),
    ("scikit-learn", sklearn.__version__),
    ("scipy", scipy.__version__),
]

versions = pd.DataFrame(version_rows, columns=["Tool", "Version"])

display(versions)
display(Markdown("**Cell 05 complete:** imports loaded and versions displayed."))


## Cell 06 - Plotly Theme Helpers

This cell sets up the chart style for the rest of the workshop. Before you run the code cell below, here is what it will do:

- define several chart theme presets instead of forcing one look
- set the active theme with one clear variable near the top of the code cell
- define `set_chart_theme(...)`, `apply_theme(...)`, and `show_metric_table(...)`
- show clickable theme cards and a preview chart so you can see the style change before later analysis charts use it

**What to check after it runs:** click a theme card. The card and preview chart should update immediately, and later cells run after that click should use the selected theme. If the notebook is viewed outside Colab or widgets/callbacks are unavailable, change `CHART_THEME_NAME` near the top of the code cell and rerun Cell 06.


In [ ]:
# Cell 06 - Plotly Theme Helpers

from html import escape
import json
from IPython.display import HTML, Markdown, display
import pandas as pd
import plotly.graph_objects as go

# Choose the default look for the rest of the notebook.
# Options: "Chart Truth", "Clean Light", "Dark Colab", "Colorblind Safe", "Institutional"
CHART_THEME_NAME = "Institutional"

CHART_THEMES = {
    "Chart Truth": {
        "ink": "#f8fafc",
        "muted": "#b6c2cf",
        "grid": "#26384f",
        "background": "#07111d",
        "panel": "#111827",
        "table_header": "#f5c45b",
        "table_header_text": "#07111d",
        "blue": "#38bdf8",
        "teal": "#2dd4bf",
        "amber": "#f5c45b",
        "rose": "#fb7185",
        "violet": "#a78bfa",
        "colorway": ["#f5c45b", "#38bdf8", "#2dd4bf", "#fb7185", "#a78bfa"],
    },
    "Clean Light": {
        "ink": "#111827",
        "muted": "#64748b",
        "grid": "#dbeafe",
        "background": "#ffffff",
        "panel": "#f8fafc",
        "table_header": "#1d4ed8",
        "table_header_text": "#ffffff",
        "blue": "#1d4ed8",
        "teal": "#0d9488",
        "amber": "#f59e0b",
        "rose": "#e11d48",
        "violet": "#7c3aed",
        "colorway": ["#1d4ed8", "#0d9488", "#f59e0b", "#e11d48", "#7c3aed"],
    },
    "Dark Colab": {
        "ink": "#e8eaed",
        "muted": "#bdc1c6",
        "grid": "#3c4043",
        "background": "#202124",
        "panel": "#2b2c30",
        "table_header": "#8ab4f8",
        "table_header_text": "#111317",
        "blue": "#8ab4f8",
        "teal": "#80cbc4",
        "amber": "#fdd663",
        "rose": "#f28b82",
        "violet": "#c58af9",
        "colorway": ["#8ab4f8", "#80cbc4", "#fdd663", "#f28b82", "#c58af9"],
    },
    "Colorblind Safe": {
        "ink": "#111827",
        "muted": "#4b5563",
        "grid": "#e5e7eb",
        "background": "#ffffff",
        "panel": "#f9fafb",
        "table_header": "#111827",
        "table_header_text": "#ffffff",
        "blue": "#0072b2",
        "teal": "#009e73",
        "amber": "#e69f00",
        "rose": "#d55e00",
        "violet": "#cc79a7",
        "colorway": ["#0072b2", "#009e73", "#e69f00", "#d55e00", "#cc79a7"],
    },
    "Institutional": {
        "ink": "#172033",
        "muted": "#64748b",
        "grid": "#d8cfc1",
        "background": "#faf7f0",
        "panel": "#f0eadf",
        "table_header": "#172033",
        "table_header_text": "#ffffff",
        "blue": "#274c77",
        "teal": "#2a9d8f",
        "amber": "#c17c0e",
        "rose": "#8f2d56",
        "violet": "#5e548e",
        "colorway": ["#274c77", "#2a9d8f", "#c17c0e", "#8f2d56", "#5e548e"],
    },
}

THEME_DESCRIPTIONS = {
    "Chart Truth": "brand-forward dark navy with gold and bright chart accents",
    "Clean Light": "bright white notebook style for maximum readability",
    "Dark Colab": "native Colab dark-mode feel with softer Google-like colors",
    "Colorblind Safe": "Okabe-Ito inspired colors for category separation",
    "Institutional": "warm research-report palette with muted professional contrast",
}


def set_chart_theme(theme_name="Institutional"):
    """Set the active workshop chart theme."""
    if theme_name not in CHART_THEMES:
        valid = ", ".join(CHART_THEMES)
        raise ValueError(f"Unknown chart theme: {theme_name}. Choose one of: {valid}.")
    globals()["ACTIVE_CHART_THEME"] = theme_name
    globals()["CHART_COLORS"] = CHART_THEMES[theme_name].copy()
    globals()["PLOTLY_COLORWAY"] = CHART_COLORS["colorway"]
    return CHART_COLORS


set_chart_theme(CHART_THEME_NAME)


def select_chart_theme(theme_name):
    """Set the active chart theme from the clickable Colab cards."""
    set_chart_theme(theme_name)
    return {"active_theme": ACTIVE_CHART_THEME, "colorway": CHART_COLORS["colorway"]}


try:
    from google.colab import output as colab_output
    colab_output.register_callback("skfolio_masterclass.select_chart_theme", select_chart_theme)
    THEME_CARD_CALLBACKS_AVAILABLE = True
except Exception:
    colab_output = None
    THEME_CARD_CALLBACKS_AVAILABLE = False


def normalize_plotly_title(title):
    """Return a centered Plotly title object from a string or existing title dict."""
    if title is None:
        return None
    if isinstance(title, dict):
        centered_title = title.copy()
    else:
        centered_title = {"text": str(title)}
    centered_title.setdefault("x", 0.5)
    centered_title.setdefault("xanchor", "center")
    centered_title.setdefault("y", 0.96)
    centered_title.setdefault("yanchor", "top")
    return centered_title


def apply_theme(fig, title=None, height=420):
    """Apply the active workshop Plotly styling and return the figure."""
    centered_title = normalize_plotly_title(title)
    title_top_margin = 90 if centered_title else 45
    fig.update_layout(
        template="plotly_white",
        title=centered_title,
        height=height,
        colorway=PLOTLY_COLORWAY,
        paper_bgcolor=CHART_COLORS["background"],
        plot_bgcolor=CHART_COLORS["background"],
        font=dict(color=CHART_COLORS["ink"], size=13),
        title_font=dict(size=18, color=CHART_COLORS["ink"]),
        margin=dict(l=60, r=35, t=title_top_margin, b=55),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
    )
    fig.update_xaxes(
        showgrid=True,
        gridcolor=CHART_COLORS["grid"],
        zeroline=False,
        title_font=dict(color=CHART_COLORS["muted"]),
        tickfont=dict(color=CHART_COLORS["muted"]),
    )
    fig.update_yaxes(
        showgrid=True,
        gridcolor=CHART_COLORS["grid"],
        zeroline=False,
        title_font=dict(color=CHART_COLORS["muted"]),
        tickfont=dict(color=CHART_COLORS["muted"]),
    )
    return fig


def show_metric_table(df, title=None, height=None):
    """Render a compact Plotly table for small summary DataFrames."""
    table_df = df.copy()
    values = [table_df[col].tolist() for col in table_df.columns]
    row_count = max(len(table_df), 1)
    table_height = height or min(560, 135 + 34 * row_count)
    column_widths = [1.1] * len(table_df.columns)
    if len(column_widths) >= 2:
        column_widths[1] = 1.8

    fig = go.Figure(
        data=[
            go.Table(
                columnwidth=column_widths,
                header=dict(
                    values=list(table_df.columns),
                    fill_color=CHART_COLORS["table_header"],
                    font=dict(color=CHART_COLORS["table_header_text"], size=12),
                    align="left",
                    height=32,
                ),
                cells=dict(
                    values=values,
                    fill_color=CHART_COLORS["panel"],
                    font=dict(color=CHART_COLORS["ink"], size=12),
                    align="left",
                    height=30,
                ),
            )
        ]
    )
    return apply_theme(fig, title=title, height=table_height)


def render_theme_cards():
    """Show clickable theme cards that update the active Colab theme."""
    theme_payload = {
        name: {
            "description": THEME_DESCRIPTIONS[name],
            "background": theme["background"],
            "panel": theme["panel"],
            "ink": theme["ink"],
            "muted": theme["muted"],
            "grid": theme["grid"],
            "table_header": theme["table_header"],
            "colorway": theme["colorway"],
        }
        for name, theme in CHART_THEMES.items()
    }
    cards = []
    for theme_name, theme in CHART_THEMES.items():
        is_active = theme_name == ACTIVE_CHART_THEME
        active_label = "Active" if is_active else "Preview"
        swatches = "".join(
            f'<span class="skfolio-theme-swatch" style="background:{color};"></span>'
            for color in theme["colorway"]
        )
        cards.append(
            f"""
            <button type="button" class="skfolio-theme-card" data-theme="{escape(theme_name)}" aria-pressed="{str(is_active).lower()}">
              <span class="skfolio-theme-card-top">
                <strong>{escape(theme_name)}</strong>
                <span class="skfolio-theme-pill">{active_label}</span>
              </span>
              <span class="skfolio-theme-description">{escape(THEME_DESCRIPTIONS[theme_name])}</span>
              <span class="skfolio-theme-swatches">{swatches}</span>
            </button>
            """
        )

    html = f"""
    <div class="skfolio-theme-selector" style="font-family:Inter,Arial,sans-serif;margin:8px 0 18px;">
      <style>
        .skfolio-theme-selector {{ color: {CHART_COLORS['ink']}; }}
        .skfolio-theme-selector-title {{ font-size:16px;font-weight:700;margin:0 0 8px; }}
        .skfolio-theme-grid {{ display:grid;grid-template-columns:repeat(auto-fit,minmax(190px,1fr));gap:10px; }}
        .skfolio-theme-card {{
          appearance:none;width:100%;min-height:138px;text-align:left;border:2px solid var(--card-border);
          border-radius:10px;padding:12px 14px;background:var(--card-bg);color:var(--card-ink);
          box-shadow:0 1px 5px rgba(0,0,0,.08);cursor:pointer;transition:transform .12s ease,border-color .12s ease,box-shadow .12s ease;
        }}
        .skfolio-theme-card:hover {{ transform:translateY(-1px);box-shadow:0 4px 12px rgba(0,0,0,.16); }}
        .skfolio-theme-card[aria-pressed="true"] {{ border-color:#f59e0b;box-shadow:0 0 0 2px rgba(245,158,11,.22),0 4px 12px rgba(0,0,0,.16); }}
        .skfolio-theme-card-top {{ display:flex;justify-content:space-between;gap:10px;align-items:center;font-size:15px; }}
        .skfolio-theme-pill {{ font-size:11px;color:var(--pill-ink);background:var(--pill-bg);padding:3px 8px;border-radius:999px; }}
        .skfolio-theme-description {{ display:block;margin:8px 0 12px;color:var(--card-muted);font-size:12px;line-height:1.35; }}
        .skfolio-theme-swatches {{ display:flex;gap:7px;align-items:center; }}
        .skfolio-theme-swatch {{ display:inline-block;width:22px;height:22px;border-radius:999px;border:1px solid rgba(0,0,0,.18); }}
        .skfolio-theme-status {{ margin:9px 0 0;color:{CHART_COLORS['muted']};font-size:12px; }}
      </style>
      <div class="skfolio-theme-selector-title">Theme choices</div>
      <div class="skfolio-theme-grid">{''.join(cards)}</div>
      <div class="skfolio-theme-status">Click a card to apply that theme to later charts in this runtime.</div>
      <script>
        (() => {{
          const script = document.currentScript;
          const root = script.closest('.skfolio-theme-selector');
          const themes = {json.dumps(theme_payload)};
          const themeOrder = {json.dumps(list(CHART_THEMES))};
          const tracesPerTheme = {len(preview_y)};
          const callbacksAvailable = {str(THEME_CARD_CALLBACKS_AVAILABLE).lower()};
          const status = root.querySelector('.skfolio-theme-status');
          const cards = Array.from(root.querySelectorAll('.skfolio-theme-card'));

          function paintCards(activeTheme) {{
            cards.forEach((card) => {{
              const themeName = card.dataset.theme;
              const theme = themes[themeName];
              const isActive = themeName === activeTheme;
              card.setAttribute('aria-pressed', isActive ? 'true' : 'false');
              card.style.setProperty('--card-bg', theme.background);
              card.style.setProperty('--card-ink', theme.ink);
              card.style.setProperty('--card-muted', theme.muted);
              card.style.setProperty('--card-border', isActive ? '#f59e0b' : theme.grid);
              card.style.setProperty('--pill-bg', isActive ? theme.table_header : theme.colorway[0]);
              card.style.setProperty('--pill-ink', theme.background);
              const pill = card.querySelector('.skfolio-theme-pill');
              if (pill) pill.textContent = isActive ? 'Active' : 'Preview';
            }});
          }}

          function updatePreviewChart(themeName) {{
            const plot = Array.from(document.querySelectorAll('.js-plotly-plot')).pop();
            if (!plot || !window.Plotly || !themes[themeName]) return;
            const themeIndex = themeOrder.indexOf(themeName);
            if (themeIndex < 0) return;
            const visible = Array(themeOrder.length * tracesPerTheme).fill(false);
            for (let i = 0; i < tracesPerTheme; i += 1) visible[themeIndex * tracesPerTheme + i] = true;
            const theme = themes[themeName];
            window.Plotly.update(
              plot,
              {{ visible, showlegend: visible }},
              {{
                title: {{ text: `Theme Preview: ${{themeName}}`, x: 0.5, xanchor: 'center', y: 0.96, yanchor: 'top' }},
                paper_bgcolor: theme.background,
                plot_bgcolor: theme.background,
                font: {{ color: theme.ink, size: 13 }},
                xaxis: {{ gridcolor: theme.grid, tickfont: {{ color: theme.muted }} }},
                yaxis: {{ gridcolor: theme.grid, tickfont: {{ color: theme.muted }} }},
                legend: {{ orientation: 'h', yanchor: 'bottom', y: 1.02, xanchor: 'center', x: 0.5 }}
              }}
            );
          }}

          async function chooseTheme(themeName) {{
            paintCards(themeName);
            updatePreviewChart(themeName);
            if (callbacksAvailable && window.google?.colab?.kernel?.invokeFunction) {{
              status.textContent = `Applying ${{themeName}} to this Colab runtime...`;
              try {{
                await window.google.colab.kernel.invokeFunction('skfolio_masterclass.select_chart_theme', [themeName], {{}});
                status.textContent = `${{themeName}} is active. Charts you run after this will use this theme.`;
              }} catch (error) {{
                status.textContent = `The preview changed, but Python did not receive the click. Set CHART_THEME_NAME = "${{themeName}}" and rerun Cell 06.`;
              }}
            }} else {{
              status.textContent = `Preview changed to ${{themeName}}. To make later Python cells use it, set CHART_THEME_NAME = "${{themeName}}" and rerun Cell 06.`;
            }}
          }}

          cards.forEach((card) => {{
            const themeName = card.dataset.theme;
            const theme = themes[themeName];
            card.style.setProperty('--card-bg', theme.background);
            card.style.setProperty('--card-ink', theme.ink);
            card.style.setProperty('--card-muted', theme.muted);
            card.style.setProperty('--card-border', card.getAttribute('aria-pressed') === 'true' ? '#f59e0b' : theme.grid);
            card.style.setProperty('--pill-bg', card.getAttribute('aria-pressed') === 'true' ? theme.table_header : theme.colorway[0]);
            card.style.setProperty('--pill-ink', theme.background);
            card.addEventListener('click', () => chooseTheme(themeName));
            card.addEventListener('keydown', (event) => {{
              if (event.key === 'Enter' || event.key === ' ') {{
                event.preventDefault();
                chooseTheme(themeName);
              }}
            }});
          }});
        }})();
      </script>
    </div>
    """
    display(HTML(html))


preview_x = ["Start", "Month 1", "Month 2", "Month 3", "Month 4"]
preview_y = {
    "Portfolio": [100, 103, 101, 108, 111],
    "Benchmark": [100, 101, 102, 104, 106],
    "Diversifier": [100, 99, 103, 102, 107],
}

preview_fig = go.Figure()
buttons = []
traces_per_theme = len(preview_y)
for theme_index, (theme_name, theme) in enumerate(CHART_THEMES.items()):
    visible = theme_name == ACTIVE_CHART_THEME
    for trace_index, (series_name, values) in enumerate(preview_y.items()):
        preview_fig.add_trace(
            go.Scatter(
                x=preview_x,
                y=values,
                mode="lines+markers",
                name=series_name,
                line=dict(
                    color=theme["colorway"][trace_index],
                    width=4 if trace_index == 0 else 3,
                    dash="dash" if series_name == "Benchmark" else None,
                ),
                marker=dict(size=8),
                visible=visible,
                showlegend=visible,
            )
        )

    visible_mask = [False] * (len(CHART_THEMES) * traces_per_theme)
    for offset in range(traces_per_theme):
        visible_mask[theme_index * traces_per_theme + offset] = True

    buttons.append(
        dict(
            label=theme_name,
            method="update",
            args=[
                {"visible": visible_mask, "showlegend": visible_mask},
                {
                    "title": normalize_plotly_title(f"Theme Preview: {theme_name}"),
                    "paper_bgcolor": theme["background"],
                    "plot_bgcolor": theme["background"],
                    "font": {"color": theme["ink"], "size": 13},
                    "xaxis": {"gridcolor": theme["grid"], "tickfont": {"color": theme["muted"]}},
                    "yaxis": {"gridcolor": theme["grid"], "tickfont": {"color": theme["muted"]}},
                    "legend": {"orientation": "h", "yanchor": "bottom", "y": 1.02, "xanchor": "center", "x": 0.5},
                },
            ],
        )
    )

preview_fig.update_layout(
    title=normalize_plotly_title(f"Theme Preview: {ACTIVE_CHART_THEME}"),
    height=410,
    paper_bgcolor=CHART_COLORS["background"],
    plot_bgcolor=CHART_COLORS["background"],
    font=dict(color=CHART_COLORS["ink"], size=13),
    margin=dict(l=60, r=35, t=95, b=55),
    updatemenus=[
        dict(
            type="dropdown",
            buttons=buttons,
            direction="down",
            x=0.035,
            xanchor="left",
            y=1.16,
            yanchor="top",
            showactive=True,
        )
    ],
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
)
preview_fig.update_xaxes(showgrid=True, gridcolor=CHART_COLORS["grid"], zeroline=False)
preview_fig.update_yaxes(showgrid=True, gridcolor=CHART_COLORS["grid"], zeroline=False)

render_theme_cards()
preview_fig.show()

display(
    Markdown(
        f"**Cell 06 complete:** `{ACTIVE_CHART_THEME}` is now the active chart theme. "
        "Click a theme card to apply that theme to this Colab runtime and update the preview. "
        "Charts you run after that click will use the selected colors. "
        "If callbacks are unavailable, set `CHART_THEME_NAME` near the top of this code cell and rerun Cell 06."
    )
)

## Cell 07 - skfolio In Plain English

skfolio is a Python toolkit for portfolio research. In this workshop, we will use it to turn asset returns into portfolios we can compare, optimize, and validate.

It helps answer practical research questions like:

- how risky was this group of ETFs?
- how did the ETFs move together?
- what portfolio weights meet the rules we set?
- how does an optimized portfolio compare with a simple baseline?
- did the result still make sense on later data?

What skfolio is not:

- not a broker or trading app
- not a market-data provider
- not a price-prediction engine
- not personalized investment advice

Think of skfolio as the research workbench. We bring it clean return data, choose a portfolio method, set constraints, and then inspect the result carefully before trusting anything.

The next cell turns these ideas into a small concept map.


In [ ]:
# Cell 08 - skfolio Concept Map Table

from IPython.display import Markdown, display
import pandas as pd
import plotly.graph_objects as go

# If the runtime restarted and Cell 06 has not been rerun yet, define a small fallback theme
# so this concept cell still works. Rerun Cell 06 when you want the full workshop theme.
if "CHART_COLORS" not in globals():
    CHART_COLORS = {
        "ink": "#1f2937",
        "muted": "#6b7280",
        "grid": "#e5e7eb",
        "background": "#ffffff",
        "panel": "#f8fafc",
        "blue": "#2563eb",
        "teal": "#0f766e",
        "amber": "#d97706",
        "rose": "#be123c",
        "violet": "#7c3aed",
    }

if "PLOTLY_COLORWAY" not in globals():
    PLOTLY_COLORWAY = [
        CHART_COLORS["blue"],
        CHART_COLORS["teal"],
        CHART_COLORS["amber"],
        CHART_COLORS["rose"],
        CHART_COLORS["violet"],
    ]

if "apply_theme" not in globals():
    def apply_theme(fig, title=None, height=420):
        fig.update_layout(
            template="plotly_white",
            title=title,
            height=height,
            colorway=PLOTLY_COLORWAY,
            paper_bgcolor=CHART_COLORS["background"],
            plot_bgcolor=CHART_COLORS["background"],
            font=dict(color=CHART_COLORS["ink"], size=13),
            title_font=dict(size=18, color=CHART_COLORS["ink"]),
            margin=dict(l=40, r=35, t=70 if title else 35, b=35),
        )
        return fig

display(
    Markdown(
        "### Cell 08 - skfolio Concept Map\n\n"
        "Before we load market data, it helps to name the main pieces of the workflow. "
        "This table is a map for the code we will build later."
    )
)

concept_map = pd.DataFrame(
    {
        "Piece": [
            "data",
            "estimator",
            "optimizer",
            "portfolio",
            "population",
            "validation",
        ],
        "Plain-English role": [
            "the return history we give to the model",
            "the tool that estimates risk, return, or relationships",
            "the tool that chooses weights under rules",
            "one set of ETF weights plus its performance results",
            "a group of portfolios we can compare side by side",
            "the check that asks whether the result still made sense later",
        ],
        "In this workshop": [
            "daily ETF returns",
            "risk and covariance estimates",
            "constrained allocation methods",
            "baseline and optimized portfolios",
            "portfolio alternatives for comparison",
            "chronological train/test review",
        ],
    }
)

concept_fig = go.Figure(
    data=[
        go.Table(
            columnwidth=[0.8, 2.2, 1.8],
            header=dict(
                values=list(concept_map.columns),
                fill_color=CHART_COLORS["ink"],
                font=dict(color="white", size=12),
                align="left",
                height=34,
            ),
            cells=dict(
                values=[concept_map[col] for col in concept_map.columns],
                fill_color=CHART_COLORS["panel"],
                font=dict(color=CHART_COLORS["ink"], size=12),
                align="left",
                height=34,
            ),
        )
    ]
)

apply_theme(
    concept_fig,
    title="How The skfolio Workflow Fits Together",
    height=420,
).show()

display(
    Markdown(
        "**What to notice:** skfolio does not replace judgment. It gives us a structured "
        "way to move from return data to portfolio choices, then compare and validate those "
        "choices before we trust them."
    )
)


## Cell 09 - skfolio Workflow From Data To Validation

Now that the main skfolio terms are on the map, we can connect them into the workflow this notebook will follow.

Think of the process as a chain:

1. **Prices** are the raw history of each ETF.
2. **Returns** make ETFs comparable by measuring percentage change.
3. **Estimates** summarize what the return history suggests about risk, return, and relationships.
4. **Optimizers** use those estimates plus our rules to choose portfolio weights.
5. **Portfolios** let us inspect the resulting weights and performance.
6. **Validation** checks whether the result still made sense on data that came later.

In skfolio language, the return matrix is usually called `X`. Each column is an asset, each row is a date, and each value is that asset's return for that date.

```text
prices
  -> returns matrix X
  -> risk and return estimates
  -> optimizer with constraints
  -> portfolio weights
  -> performance review
  -> train/test validation
```

**What to notice:** the optimizer is only one step in the workflow. A portfolio can look impressive during construction and still fail validation later. That is why this workshop keeps the data preparation, constraints, comparison, and validation steps visible instead of hiding them behind one final answer.


## Cell 10 - Why ETFs Instead Of Individual Stocks

Before we load the dataset, it is worth explaining why this workshop starts with ETFs as the teaching universe.

This is not because ETFs are the only good portfolio building block. Individual stocks can absolutely be combined into custom baskets. In fact, a later notebook could explore that path: selecting stocks, grouping them into exposures, and creating our own ETF-like basket before passing the returns into skfolio.

For Notebook 01, though, custom stock selection would add a second problem before we have learned the first one. We would need to decide which companies belong in the universe, how to handle sector balance, corporate actions, survivorship bias, missing histories, thin trading, and company-specific news. Those are real and important topics, but they can distract from the core skfolio workflow.

ETFs give us a cleaner first layer. They are already packaged exposures with tickers, price histories, liquidity profiles, and often published holdings or mandates. That lets us focus on the portfolio research process: validate the data, build returns, inspect relationships, set constraints, and later compare portfolios.

Why ETFs fit this first notebook:

- **Cleaner exposure labels:** an ETF can represent a market, sector, bond category, commodity exposure, factor, region, or investment style.
- **Built-in diversification:** many ETFs already hold baskets of securities, so we can study allocation across exposures without managing every underlying stock on day one.
- **Practical data and liquidity:** large ETFs often have accessible price histories and active trading, which makes data checks and trading assumptions easier to discuss.
- **Less security-selection noise:** we can postpone company-picking decisions and focus first on how skfolio expects returns, constraints, and validation to work.
- **Still realistic portfolio choices:** ETFs still force real tradeoffs across risk, correlation, concentration, costs, and liquidity.

ETFs still have tradeoffs. They can carry fees, tracking differences, concentration risk, tax considerations, changing holdings, and liquidity differences. Some ETFs are broad and simple; others are narrow, leveraged, complex, or thinly traded.

**What to notice:** ETFs are not "better" than individual stocks for every purpose. They are the right first teaching universe because they let us learn the portfolio workflow before we add the harder problem of building our own stock universe or custom basket.

In [ ]:
# Cell 11 - Exposure Comparison Table

from IPython.display import Markdown, display
import pandas as pd
import plotly.graph_objects as go

# Small fallback theme in case this cell is run after a runtime reset.
if "CHART_COLORS" not in globals():
    CHART_COLORS = {
        "ink": "#1f2937",
        "muted": "#6b7280",
        "grid": "#e5e7eb",
        "background": "#ffffff",
        "panel": "#f8fafc",
        "blue": "#2563eb",
        "teal": "#0f766e",
        "amber": "#d97706",
        "rose": "#be123c",
        "violet": "#7c3aed",
    }

if "PLOTLY_COLORWAY" not in globals():
    PLOTLY_COLORWAY = [
        CHART_COLORS["blue"],
        CHART_COLORS["teal"],
        CHART_COLORS["amber"],
        CHART_COLORS["rose"],
        CHART_COLORS["violet"],
    ]

if "apply_theme" not in globals():
    def apply_theme(fig, title=None, height=420):
        fig.update_layout(
            template="plotly_white",
            title=title,
            height=height,
            colorway=PLOTLY_COLORWAY,
            paper_bgcolor=CHART_COLORS["background"],
            plot_bgcolor=CHART_COLORS["background"],
            font=dict(color=CHART_COLORS["ink"], size=13),
            title_font=dict(size=18, color=CHART_COLORS["ink"]),
            margin=dict(l=40, r=35, t=70 if title else 35, b=35),
        )
        return fig


display(
    Markdown(
        "### Cell 11 - Exposure Comparison Table\n\n"
        "This table compares the kinds of building blocks we could give to a portfolio "
        "optimizer. The key distinction is whether we are mainly studying one company "
        "or choosing between broader market exposures."
    )
)

exposure_comparison = pd.DataFrame(
    [
        {
            "Building block": "Individual stock",
            "Example exposure": "One company",
            "What it represents": "A single business and its company-specific risks",
            "Main research question": "Do I understand this company well enough?",
            "Allocation role": "Useful for security selection, but noisier for a first allocation lesson",
        },
        {
            "Building block": "Sector ETF",
            "Example exposure": "Technology, health care, energy",
            "What it represents": "A basket of companies in one industry group",
            "Main research question": "How much sector tilt should the portfolio accept?",
            "Allocation role": "Adds targeted exposure while keeping the input list manageable",
        },
        {
            "Building block": "Broad equity ETF",
            "Example exposure": "U.S. stocks, international stocks, global stocks",
            "What it represents": "A large slice of the equity market",
            "Main research question": "How much equity risk should the portfolio carry?",
            "Allocation role": "Works well as a core growth exposure",
        },
        {
            "Building block": "Bond ETF",
            "Example exposure": "Treasury bonds, aggregate bonds, short-term bonds",
            "What it represents": "A slice of the fixed-income market",
            "Main research question": "How much stability or rate sensitivity is useful?",
            "Allocation role": "Often used to balance equity risk and reduce portfolio swings",
        },
        {
            "Building block": "Commodity ETF",
            "Example exposure": "Gold, broad commodities",
            "What it represents": "A commodity-linked return stream",
            "Main research question": "Does this diversify the rest of the portfolio?",
            "Allocation role": "Can add diversification, but needs careful risk and liquidity review",
        },
    ]
)

fig = go.Figure(
    data=[
        go.Table(
            columnwidth=[1.0, 1.3, 2.0, 2.0, 2.2],
            header=dict(
                values=list(exposure_comparison.columns),
                fill_color=CHART_COLORS["ink"],
                font=dict(color="white", size=12),
                align="left",
                height=34,
            ),
            cells=dict(
                values=[exposure_comparison[col].tolist() for col in exposure_comparison.columns],
                fill_color=CHART_COLORS["panel"],
                font=dict(color=CHART_COLORS["ink"], size=12),
                align="left",
                height=54,
            ),
        )
    ]
)

fig = apply_theme(fig, title="Portfolio Inputs: From Company Bets To Market Exposures", height=430)
fig.update_layout(margin=dict(l=30, r=30, t=70, b=30))
fig.show()

display(
    Markdown(
        "**What to notice:** the same optimizer can work with many asset types, but the "
        "interpretation changes. This workshop starts with ETFs because they make the "
        "allocation question clearer: we can compare exposures, constraints, and risk "
        "without first doing a full company-by-company research process."
    )
)


## Cell 12 - What To Notice About ETFs As Inputs

The important idea is not that ETFs are the only assets skfolio can use. skfolio works with return data, so the input could come from ETFs, stocks, mutual funds, crypto assets, private indexes, strategy returns, or another clean return matrix.

ETFs are useful here because they make the first learning problem easier to see:

- **The rows and columns stay interpretable.** Each ticker usually points to a market, sector, bond category, commodity exposure, factor, region, or investment style.
- **The allocation question is clearer.** We can ask how much to allocate to each exposure before we ask which individual companies deserve research time.
- **The universe stays manageable.** A small ETF list lets us inspect risk, liquidity, correlations, constraints, and validation without drowning in hundreds of tickers.
- **The tradeoffs are still real.** Fees, liquidity, tracking differences, concentration, tax considerations, and changing holdings can still matter.

Later, we can make the problem more ambitious. Instead of using ETFs as prebuilt exposure baskets, we could select individual stocks, group them into themes or sectors, and create our own custom basket. The skfolio workflow would still start from returns, but the universe-design responsibility would move from the ETF issuer to us.

**What to notice:** the asset universe is part of the portfolio design. Before an optimizer chooses weights, we choose what it is allowed to choose from. In the next section, we load the public ETF dataset that will become our first teaching universe.

## Cell 13 - The Public ETF Dataset We Will Use

Now we are ready to load the teaching dataset.

This workshop uses a public ETF dataset hosted on Hugging Face. You do not need a broker login, API key, paid data subscription, or private credentials to run the next few cells. Colab will download the dataset files directly into the notebook runtime.

The dataset is split into a few files so each part has a clear job:

| File | What it contains | Why we need it |
|---|---|---|
| `etf_daily_prices.parquet` | Daily ETF price and volume history | This becomes the raw market history we convert into returns. |
| `instrument_metadata.parquet` | Descriptive information for each ETF | This helps us label, filter, and understand the ETF universe. |
| `etf_liquidity_365d.parquet` | Recent liquidity summaries | This helps us avoid building examples around ETFs that are too thinly traded. |
| `manifest.json` | Dataset notes, file metadata, and version-style details | This gives us a quick way to check what we downloaded. |

We will keep the loading process visible instead of hiding it in one large function. That makes it easier to see where the data comes from, what shape it has, and how we check it before using it in skfolio.

**What to notice:** data loading is part of the research workflow, not just setup. A portfolio optimizer can only work with the assets and history we give it, so the next cells download, load, and validate the dataset before any portfolio construction begins.


## Cell 14 - Dataset Config

This cell creates the dataset settings that the download and loading cells will reuse. Before you run the code cell below, here is what it will do:

- set the public Hugging Face dataset repository name
- list the four dataset files we expect to download
- set the temporary Colab folder where the files will be stored
- create a `dataset_config` table so the settings are easy to inspect
- show a wrapped summary and a dataset page link

**Runtime note:** `/content/skfolio_etf_data` is temporary Colab runtime storage. If the runtime restarts, rerun this cell before rerunning Cell 15.

**What to check after it runs:** you should see the dataset repository, file names, cache folder, and a short completion message. The text should wrap cleanly without being cut off.


In [ ]:
# Cell 14 - Dataset Config

from html import escape
from pathlib import Path

from IPython.display import HTML, Markdown, display
import pandas as pd


DATASET_REPO_ID = "thecharttruth/etf-data"
DATASET_REPO_TYPE = "dataset"
DATASET_URL = f"https://huggingface.co/datasets/{DATASET_REPO_ID}"

DATASET_FILES = {
    "prices": "etf_daily_prices.parquet",
    "metadata": "instrument_metadata.parquet",
    "liquidity": "etf_liquidity_365d.parquet",
    "manifest": "manifest.json",
}

DATA_CACHE_DIR = Path("/content/skfolio_etf_data")

config_rows = [
    {
        "Setting": "Dataset repository",
        "Value": DATASET_REPO_ID,
        "Why it matters": "This is the public Hugging Face dataset we will download from.",
    },
    {
        "Setting": "Repository type",
        "Value": DATASET_REPO_TYPE,
        "Why it matters": "Hugging Face separates datasets from model repositories.",
    },
    {
        "Setting": "Local cache folder",
        "Value": str(DATA_CACHE_DIR),
        "Why it matters": "Colab stores downloaded files here for this runtime session.",
    },
    {
        "Setting": "Price file",
        "Value": DATASET_FILES["prices"],
        "Why it matters": "Raw daily ETF prices and volume; this is the source for returns.",
    },
    {
        "Setting": "Metadata file",
        "Value": DATASET_FILES["metadata"],
        "Why it matters": "ETF descriptions and labels for understanding the universe.",
    },
    {
        "Setting": "Liquidity file",
        "Value": DATASET_FILES["liquidity"],
        "Why it matters": "Recent trading-activity summary for practical filtering.",
    },
    {
        "Setting": "Manifest file",
        "Value": DATASET_FILES["manifest"],
        "Why it matters": "Dataset notes and file-level details for quick checks.",
    },
]

dataset_config = pd.DataFrame(config_rows)


def render_config_summary(rows):
    """Render a small, wrapped config summary that stays readable in Colab."""
    cards = []
    for row in rows:
        cards.append(
            f"""
            <div class="config-card">
                <div class="config-setting">{escape(row["Setting"])}</div>
                <div class="config-value">{escape(row["Value"])}</div>
                <div class="config-why">{escape(row["Why it matters"])}</div>
            </div>
            """
        )

    return HTML(
        f"""
        <style>
            .dataset-config-summary {{
                max-width: 980px;
                display: grid;
                gap: 8px;
                margin: 12px 0 10px;
                font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
            }}
            .config-card {{
                display: grid;
                grid-template-columns: minmax(150px, 0.85fr) minmax(190px, 1fr) minmax(260px, 1.65fr);
                gap: 12px;
                align-items: start;
                padding: 10px 12px;
                border: 1px solid rgba(127, 127, 127, 0.28);
                border-radius: 6px;
                background: rgba(127, 127, 127, 0.08);
            }}
            .config-setting {{
                font-weight: 700;
            }}
            .config-value {{
                font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;
                overflow-wrap: anywhere;
            }}
            .config-why {{
                line-height: 1.35;
            }}
            @media (max-width: 760px) {{
                .config-card {{
                    grid-template-columns: 1fr;
                    gap: 4px;
                }}
                .config-setting {{
                    margin-top: 2px;
                }}
            }}
        </style>
        <div class="dataset-config-summary">
            {"".join(cards)}
        </div>
        """
    )


display(render_config_summary(config_rows))
display(Markdown(f"Dataset page: [{DATASET_REPO_ID}]({DATASET_URL})"))
display(Markdown("**Cell 14 complete:** dataset configuration is ready for download."))


## Cell 15 - Download Dataset Files

This is the first cell that brings dataset files into the Colab runtime. Before you run the code cell below, here is what it will do:

- download the four public dataset files named in Cell 14 into `/content/skfolio_etf_data`
- reuse files that are already available in this Colab runtime
- keep the resolved file paths in `downloaded_paths`
- create a small `download_summary` table for later inspection
- avoid Hugging Face token prompts because the dataset is public

**Runtime note:** these files live in temporary Colab runtime storage. If the runtime restarts, rerun Cell 14 and the code cell below to refresh them.

**What to check after it runs:** the output should list four files and show either `Downloaded` or `Already available` for each one.


In [ ]:
# Cell 15 - Download Dataset Files

import contextlib
from html import escape
import io
import logging
import os
from pathlib import Path
import warnings

from IPython.display import HTML, Markdown, display
from huggingface_hub import hf_hub_download
import pandas as pd


os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub.utils._http").setLevel(logging.ERROR)

required_config_names = [
    "DATASET_REPO_ID",
    "DATASET_REPO_TYPE",
    "DATASET_URL",
    "DATASET_FILES",
    "DATA_CACHE_DIR",
]

missing_config = [name for name in required_config_names if name not in globals()]
if missing_config:
    missing_text = ", ".join(missing_config)
    raise RuntimeError(
        "Cell 15 needs the dataset settings from Cell 14 first. "
        f"Run Cell 14, then run this cell again. Missing: {missing_text}"
    )

DATA_CACHE_DIR = Path(DATA_CACHE_DIR)
DATA_CACHE_DIR.mkdir(parents=True, exist_ok=True)

downloaded_paths = {}
download_rows = []

try:
    for file_key, filename in DATASET_FILES.items():
        before_exists = (DATA_CACHE_DIR / filename).exists()
        with warnings.catch_warnings():
            warnings.filterwarnings(
                "ignore",
                message=r".*HF_TOKEN.*",
                category=UserWarning,
                module=r"huggingface_hub.*",
            )
            with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                local_path = Path(
                    hf_hub_download(
                        repo_id=DATASET_REPO_ID,
                        filename=filename,
                        repo_type=DATASET_REPO_TYPE,
                        local_dir=DATA_CACHE_DIR,
                        token=False,
                    )
                )

        downloaded_paths[file_key] = local_path
        download_rows.append(
            {
                "Dataset part": file_key,
                "File": filename,
                "Local path": str(local_path),
                "Size MB": round(local_path.stat().st_size / (1024 * 1024), 2),
                "Status": "Already available" if before_exists else "Downloaded",
            }
        )
except Exception as exc:
    display(
        Markdown(
            "#### Download did not complete\n\n"
            "Colab could not download one or more dataset files. Check that the runtime "
            "is connected and that the dataset page is reachable:\n\n"
            f"- Dataset page: [{DATASET_REPO_ID}]({DATASET_URL})\n"
            f"- Files expected: `{', '.join(DATASET_FILES.values())}`\n\n"
            "After checking that, rerun this cell."
        )
    )
    raise RuntimeError("Dataset download failed. See the message above for the recovery steps.") from exc

download_summary = pd.DataFrame(download_rows)


def render_download_summary(rows):
    cards = []
    for row in rows:
        cards.append(
            f"""
            <div class="download-card">
                <div class="download-part">{escape(str(row["Dataset part"]))}</div>
                <div class="download-file">{escape(str(row["File"]))}</div>
                <div class="download-path">{escape(str(row["Local path"]))}</div>
                <div class="download-size">{escape(str(row["Size MB"]))} MB</div>
                <div class="download-status">{escape(str(row["Status"]))}</div>
            </div>
            """
        )

    return HTML(
        f"""
        <style>
            .download-summary {{
                max-width: 980px;
                display: grid;
                gap: 8px;
                margin: 12px 0 10px;
                font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
            }}
            .download-card {{
                display: grid;
                grid-template-columns: minmax(105px, 0.65fr) minmax(190px, 1fr) minmax(260px, 1.45fr) minmax(70px, 0.4fr) minmax(120px, 0.55fr);
                gap: 12px;
                align-items: start;
                padding: 10px 12px;
                border: 1px solid rgba(127, 127, 127, 0.28);
                border-radius: 6px;
                background: rgba(127, 127, 127, 0.08);
            }}
            .download-part {{
                font-weight: 700;
                text-transform: capitalize;
            }}
            .download-file,
            .download-path {{
                font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;
                overflow-wrap: anywhere;
            }}
            .download-size {{
                text-align: right;
                white-space: nowrap;
            }}
            .download-status {{
                font-weight: 700;
            }}
            @media (max-width: 820px) {{
                .download-card {{
                    grid-template-columns: 1fr;
                    gap: 4px;
                }}
                .download-size {{
                    text-align: left;
                }}
            }}
        </style>
        <div class="download-summary">
            {"".join(cards)}
        </div>
        """
    )


display(render_download_summary(download_rows))
display(
    Markdown(
        f"Downloaded or confirmed `{len(downloaded_paths)}` files in `{DATA_CACHE_DIR}`. "
        "The next cell will load these files into pandas.\n\n"
        "**Cell 15 complete:** dataset files are downloaded and ready to load."
    )
)


## Cell 16 - Load Parquet And Manifest

This cell turns the downloaded files from Cell 15 into Python objects we can inspect and reuse. Before you run the code cell below, here is what it will do:

- read the daily price history into `prices`
- read ETF descriptions into `metadata`
- read recent liquidity summaries into `liquidity`
- read dataset notes from `manifest.json` into `manifest`
- create a compact `dataset_load_summary` table with row counts, columns, and date coverage
- show one simple raw-price sanity chart so the dataset feels real before we transform it

**Runtime note:** this cell expects Cell 14 and Cell 15 to have run in the current runtime. If the runtime restarted, rerun Cell 14 and Cell 15 first.

**What to check after it runs:** the output should show three loaded tables, a manifest entry, a date range for the price history, and one readable close-price chart for a high-coverage ETF.


In [ ]:
# Cell 16 - Load Parquet And Manifest

import json
from html import escape
from pathlib import Path

from IPython.display import HTML, Markdown, display
import pandas as pd
import plotly.graph_objects as go


required_data_names = [
    "DATASET_FILES",
    "DATA_CACHE_DIR",
]

missing_data_names = [name for name in required_data_names if name not in globals()]
if missing_data_names:
    missing_text = ", ".join(missing_data_names)
    raise RuntimeError(
        "Cell 16 needs the dataset settings from Cell 14 first. "
        f"Run Cell 14 and Cell 15, then run this cell again. Missing: {missing_text}"
    )

DATA_CACHE_DIR = Path(DATA_CACHE_DIR)


def resolve_dataset_path(file_key):
    """Return the local path for a downloaded dataset file."""
    if "downloaded_paths" in globals() and file_key in downloaded_paths:
        candidate_path = Path(downloaded_paths[file_key])
    else:
        candidate_path = DATA_CACHE_DIR / DATASET_FILES[file_key]

    if not candidate_path.exists():
        raise FileNotFoundError(
            f"Could not find {DATASET_FILES[file_key]} at {candidate_path}. "
            "Rerun Cell 15 to download the dataset files, then rerun Cell 16."
        )

    return candidate_path


dataset_paths = {file_key: resolve_dataset_path(file_key) for file_key in DATASET_FILES}

prices = pd.read_parquet(dataset_paths["prices"])
metadata = pd.read_parquet(dataset_paths["metadata"])
liquidity = pd.read_parquet(dataset_paths["liquidity"])

with open(dataset_paths["manifest"], "r", encoding="utf-8") as manifest_file:
    manifest = json.load(manifest_file)

if "date" in prices.columns:
    prices["date"] = pd.to_datetime(prices["date"], errors="coerce")


def describe_date_range(frame):
    if "date" not in frame.columns:
        return "not date-indexed"

    valid_dates = frame["date"].dropna()
    if valid_dates.empty:
        return "date column has no parsed values"

    return f"{valid_dates.min().date()} to {valid_dates.max().date()}"


def describe_columns(columns, limit=7):
    column_names = [str(column) for column in columns]
    if len(column_names) <= limit:
        return ", ".join(column_names)
    return ", ".join(column_names[:limit]) + f", and {len(column_names) - limit} more"


manifest_keys = list(manifest.keys()) if isinstance(manifest, dict) else []
manifest_label = ", ".join(manifest_keys[:8]) if manifest_keys else type(manifest).__name__
if len(manifest_keys) > 8:
    manifest_label += f", and {len(manifest_keys) - 8} more"

dataset_load_summary = pd.DataFrame(
    [
        {
            "Dataset part": "prices",
            "Python object": "prices",
            "Rows or entries": f"{len(prices):,} rows",
            "Columns or keys": describe_columns(prices.columns),
            "Date coverage": describe_date_range(prices),
        },
        {
            "Dataset part": "metadata",
            "Python object": "metadata",
            "Rows or entries": f"{len(metadata):,} rows",
            "Columns or keys": describe_columns(metadata.columns),
            "Date coverage": describe_date_range(metadata),
        },
        {
            "Dataset part": "liquidity",
            "Python object": "liquidity",
            "Rows or entries": f"{len(liquidity):,} rows",
            "Columns or keys": describe_columns(liquidity.columns),
            "Date coverage": describe_date_range(liquidity),
        },
        {
            "Dataset part": "manifest",
            "Python object": "manifest",
            "Rows or entries": f"{len(manifest_keys):,} top-level keys"
            if manifest_keys
            else "loaded JSON",
            "Columns or keys": manifest_label,
            "Date coverage": "not a table",
        },
    ]
)


def render_load_summary(rows):
    cards = []
    for row in rows:
        cards.append(
            f"""
            <div class="load-card">
                <div class="load-part">{escape(str(row["Dataset part"]))}</div>
                <div class="load-object">{escape(str(row["Python object"]))}</div>
                <div class="load-size">{escape(str(row["Rows or entries"]))}</div>
                <div class="load-columns">{escape(str(row["Columns or keys"]))}</div>
                <div class="load-dates">{escape(str(row["Date coverage"]))}</div>
            </div>
            """
        )

    return HTML(
        f"""
        <style>
            .load-summary {{
                max-width: 980px;
                display: grid;
                gap: 8px;
                margin: 12px 0 10px;
                font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
            }}
            .load-card {{
                display: grid;
                grid-template-columns: minmax(105px, 0.65fr) minmax(110px, 0.6fr) minmax(120px, 0.7fr) minmax(260px, 1.55fr) minmax(170px, 0.9fr);
                gap: 12px;
                align-items: start;
                padding: 10px 12px;
                border: 1px solid rgba(127, 127, 127, 0.28);
                border-radius: 6px;
                background: rgba(127, 127, 127, 0.08);
            }}
            .load-part {{
                font-weight: 700;
                text-transform: capitalize;
            }}
            .load-object {{
                font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;
                font-weight: 700;
            }}
            .load-size {{
                white-space: nowrap;
            }}
            .load-columns,
            .load-dates {{
                overflow-wrap: anywhere;
                line-height: 1.35;
            }}
            @media (max-width: 860px) {{
                .load-card {{
                    grid-template-columns: 1fr;
                    gap: 4px;
                }}
            }}
        </style>
        <div class="load-summary">
            {"".join(cards)}
        </div>
        """
    )


display(render_load_summary(dataset_load_summary.to_dict("records")))

price_sanity_source = prices.loc[:, ["date", "symbol", "close"]].copy()
price_sanity_source["date"] = pd.to_datetime(price_sanity_source["date"], errors="coerce")
price_sanity_source["symbol"] = price_sanity_source["symbol"].astype(str).str.strip().str.upper()
price_sanity_source["close"] = pd.to_numeric(price_sanity_source["close"], errors="coerce")
price_sanity_source = price_sanity_source.dropna(subset=["date", "symbol", "close"])
price_sanity_source = price_sanity_source.loc[price_sanity_source["close"] > 0]

if not price_sanity_source.empty:
    sanity_symbol = price_sanity_source.groupby("symbol")["date"].count().sort_values(ascending=False).index[0]
    sanity_series = price_sanity_source.loc[price_sanity_source["symbol"].eq(sanity_symbol)].sort_values("date")

    fig = go.Figure(
        go.Scatter(
            x=sanity_series["date"],
            y=sanity_series["close"],
            mode="lines",
            name=sanity_symbol,
            hovertemplate=f"<b>{sanity_symbol}</b><br>%{{x|%Y-%m-%d}}<br>Close: %{{y:,.2f}}<extra></extra>",
        )
    )
    chart_title = dict(
        text=f"Raw Close Price Sanity Check: {sanity_symbol}",
        x=0.5,
        xanchor="center",
        y=0.96,
        yanchor="top",
        font=dict(size=18),
    )
    fig.update_layout(
        title=chart_title,
        xaxis_title="Date",
        yaxis_title="Close price",
        height=420,
        margin=dict(l=70, r=40, t=80, b=60),
    )
    if "apply_theme" in globals():
        fig = apply_theme(fig, title=chart_title, height=420)
        fig.update_layout(margin=dict(l=70, r=40, t=80, b=60))
    fig.show()

    display(
        Markdown(
            f"**Learner checkpoint:** `prices`, `metadata`, `liquidity`, and `manifest` are loaded. "
            f"The raw chart for `{sanity_symbol}` is only a sanity check, not a performance conclusion. "
            "The next cell validates the columns before any modeling work begins."
        )
    )
else:
    display(
        Markdown(
            "**Learner checkpoint:** the files loaded, but no positive close-price rows were available for the quick sanity chart. "
            "Cell 17 will still validate the schema and stop clearly if required fields are missing."
        )
    )

display(
    Markdown(
        "**Cell 16 complete:** dataset files are loaded into pandas and JSON objects. "
        "The next cell will validate the required columns before we transform prices into returns."
    )
)


## Cell 17 - Validate Dataset Schema

This cell checks that the loaded dataset has the columns the rest of the notebook expects. Before you run the code cell below, here is what it will do:

- confirm that `prices`, `metadata`, and `liquidity` exist from Cell 16
- check that the price table includes `date`, `symbol`, `close`, and `volume`
- check that the metadata and liquidity tables each include `symbol`
- create a compact `schema_validation` table showing pass/fail status for each dataset part
- create `dataset_field_guide`, a plain-English mini data dictionary for the fields we rely on most
- stop with a clear message if a required column is missing

**Runtime note:** this cell expects Cell 16 to have loaded the parquet files into pandas DataFrames. If the runtime restarted, rerun Cell 14 through Cell 16 first.

**What to check after it runs:** every validation row should show `Pass`, and the field guide should make the important dataset columns understandable before the notebook starts transforming them.


In [ ]:
# Cell 17 - Validate Dataset Schema

from html import escape

from IPython.display import HTML, Markdown, display
import pandas as pd


required_loaded_names = ["prices", "metadata", "liquidity"]
missing_loaded_names = [name for name in required_loaded_names if name not in globals()]
if missing_loaded_names:
    missing_text = ", ".join(missing_loaded_names)
    raise RuntimeError(
        "Cell 17 needs the loaded tables from Cell 16 first. "
        f"Run Cell 16, then run this cell again. Missing: {missing_text}"
    )

required_schema = {
    "prices": {
        "frame": prices,
        "required columns": ["date", "symbol", "close", "volume"],
        "why": "prices become the return matrix and need date, ticker, price, and volume fields",
    },
    "metadata": {
        "frame": metadata,
        "required columns": ["symbol"],
        "why": "metadata must join back to each ETF ticker",
    },
    "liquidity": {
        "frame": liquidity,
        "required columns": ["symbol"],
        "why": "liquidity must join back to each ETF ticker",
    },
}

schema_rows = []
missing_schema = {}

for dataset_name, details in required_schema.items():
    frame = details["frame"]
    required_columns = details["required columns"]

    if not isinstance(frame, pd.DataFrame):
        raise TypeError(
            f"`{dataset_name}` should be a pandas DataFrame, but found {type(frame).__name__}. "
            "Rerun Cell 16, then rerun this cell."
        )

    present_columns = set(frame.columns)
    missing_columns = [column for column in required_columns if column not in present_columns]
    extra_count = max(len(frame.columns) - len(required_columns), 0)

    schema_rows.append(
        {
            "Dataset part": dataset_name,
            "Rows": f"{len(frame):,}",
            "Required columns": ", ".join(required_columns),
            "Missing columns": ", ".join(missing_columns) if missing_columns else "None",
            "Other columns": f"{extra_count:,}",
            "Status": "Pass" if not missing_columns else "Needs attention",
            "Why this check matters": details["why"],
        }
    )

    if missing_columns:
        missing_schema[dataset_name] = missing_columns

schema_validation = pd.DataFrame(schema_rows)


def render_schema_validation(rows):
    cards = []
    for row in rows:
        status = str(row["Status"])
        status_class = "schema-pass" if status == "Pass" else "schema-warning"
        cards.append(
            f"""
            <div class="schema-card">
                <div class="schema-part">{escape(str(row["Dataset part"]))}</div>
                <div class="schema-rows">{escape(str(row["Rows"]))} rows</div>
                <div class="schema-required">{escape(str(row["Required columns"]))}</div>
                <div class="schema-missing">{escape(str(row["Missing columns"]))}</div>
                <div class="{status_class}">{escape(status)}</div>
                <div class="schema-why">{escape(str(row["Why this check matters"]))}</div>
            </div>
            """
        )

    return HTML(
        f"""
        <style>
            .schema-summary {{
                max-width: 980px;
                display: grid;
                gap: 8px;
                margin: 12px 0 10px;
                font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
            }}
            .schema-card {{
                display: grid;
                grid-template-columns: minmax(95px, 0.6fr) minmax(90px, 0.5fr) minmax(220px, 1.25fr) minmax(130px, 0.75fr) minmax(90px, 0.55fr) minmax(230px, 1.35fr);
                gap: 12px;
                align-items: start;
                padding: 10px 12px;
                border: 1px solid rgba(127, 127, 127, 0.28);
                border-radius: 6px;
                background: rgba(127, 127, 127, 0.08);
            }}
            .schema-part {{
                font-weight: 700;
                text-transform: capitalize;
            }}
            .schema-required,
            .schema-missing {{
                font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;
                overflow-wrap: anywhere;
            }}
            .schema-pass {{
                font-weight: 700;
                color: #2e7d32;
            }}
            .schema-warning {{
                font-weight: 700;
                color: #b26a00;
            }}
            .schema-why {{
                line-height: 1.35;
            }}
            @media (max-width: 900px) {{
                .schema-card {{
                    grid-template-columns: 1fr;
                    gap: 4px;
                }}
            }}
        </style>
        <div class="schema-summary">
            {"".join(cards)}
        </div>
        """
    )


def render_compact_table(frame, title, note=None):
    header = "".join(f"<th>{escape(str(column))}</th>" for column in frame.columns)
    rows = []
    for row in frame.itertuples(index=False):
        cells = "".join(f"<td>{escape(str(value))}</td>" for value in row)
        rows.append(f"<tr>{cells}</tr>")
    note_html = f"<p class='guide-note'>{escape(note)}</p>" if note else ""
    return HTML(
        f"""
        <style>
          .guide-wrap {{font-family: Arial, sans-serif; line-height: 1.45; max-width: 1100px;}}
          .guide-note {{color: #374151; margin: 6px 0 14px;}}
          .guide-scroll {{overflow-x: auto; margin: 8px 0 18px;}}
          .guide-table {{border-collapse: collapse; width: 100%; min-width: 760px; font-size: 14px; background: #ffffff;}}
          .guide-table th {{text-align: left; border-bottom: 2px solid #111827; padding: 9px 10px; background: #111827; color: #ffffff; font-weight: 700; white-space: nowrap;}}
          .guide-table td {{border-bottom: 1px solid #e5e7eb; padding: 9px 10px; color: #111827; background: #ffffff; vertical-align: top;}}
          .guide-table tbody tr:nth-child(even) td {{background: #f9fafb;}}
        </style>
        <div class="guide-wrap">
          <h3>{escape(title)}</h3>
          {note_html}
          <div class="guide-scroll"><table class="guide-table"><thead><tr>{header}</tr></thead><tbody>{''.join(rows)}</tbody></table></div>
        </div>
        """
    )


display(render_schema_validation(schema_validation.to_dict("records")))

if missing_schema:
    missing_lines = [
        f"- `{dataset_name}` is missing: `{', '.join(columns)}`"
        for dataset_name, columns in missing_schema.items()
    ]
    display(
        Markdown(
            "#### Schema validation needs attention\n\n"
            + "\n".join(missing_lines)
            + "\n\nRerun Cell 15 and Cell 16. If the same issue remains, the dataset file may have changed."
        )
    )
    raise RuntimeError("Dataset schema validation failed. See the missing-column list above.")

dataset_field_guide = pd.DataFrame(
    [
        {
            "Dataset part": "prices",
            "Field or idea": "date",
            "Plain-English meaning": "The trading day for each observation.",
            "Why it matters later": "Dates become the row index for price and return matrices.",
        },
        {
            "Dataset part": "prices",
            "Field or idea": "symbol",
            "Plain-English meaning": "The ETF ticker.",
            "Why it matters later": "Symbols become portfolio asset columns.",
        },
        {
            "Dataset part": "prices",
            "Field or idea": "close",
            "Plain-English meaning": "The daily closing price used for the teaching return calculation.",
            "Why it matters later": "Cell 25 converts close prices into linear returns.",
        },
        {
            "Dataset part": "prices",
            "Field or idea": "volume",
            "Plain-English meaning": "How many shares traded that day.",
            "Why it matters later": "Volume helps build liquidity context before choosing examples.",
        },
        {
            "Dataset part": "metadata",
            "Field or idea": "descriptive columns",
            "Plain-English meaning": "Labels that help explain what each ETF represents.",
            "Why it matters later": "Metadata keeps tables readable and helps learners spot duplicated or confusing exposures.",
        },
        {
            "Dataset part": "liquidity",
            "Field or idea": "recent trading activity",
            "Plain-English meaning": "Summary fields such as share volume or dollar volume over a recent window.",
            "Why it matters later": "Liquidity rankings help avoid building examples around very thinly traded ETFs.",
        },
        {
            "Dataset part": "manifest",
            "Field or idea": "dataset notes",
            "Plain-English meaning": "File-level context about the teaching dataset.",
            "Why it matters later": "The manifest makes the data source easier to audit and explain.",
        },
    ]
)

display(
    render_compact_table(
        dataset_field_guide,
        title="Dataset Field Guide",
        note="This is the small vocabulary we need before asking skfolio to learn from ETF returns.",
    )
)

display(
    Markdown(
        "**Learner checkpoint:** the dataset objects exist, required columns are present, and the key fields have names you can reason about. "
        "From here forward, the notebook can focus on coverage, liquidity, prices, and returns instead of guessing what the raw files contain."
    )
)

display(
    Markdown(
        "**Cell 17 complete:** required dataset columns are present and `dataset_field_guide` is ready for reference. "
        "The next cell can summarize dataset coverage with more confidence."
    )
)


## Cell 18 - Dataset Coverage Summary

This cell summarizes how much price history is available for each ETF in the dataset. Before you run the code cell below, here is what it will do:

- count the total rows, ETF symbols, and trading dates in `prices`
- create `symbol_coverage`, a table with observations, first date, last date, and coverage share for each symbol
- create `dataset_coverage_summary`, a compact table with the dataset-wide coverage numbers
- show the ETFs with the most available observations in a horizontal Plotly bar chart
- keep the output focused on coverage, not portfolio performance

**Runtime note:** this cell expects Cell 16 and Cell 17 to have run successfully. If the runtime restarted, rerun Cell 14 through Cell 17 first.

**What to check after it runs:** the chart should make it easy to see which ETFs have the longest usable history. Longer history does not automatically make an ETF better, but it usually gives the optimizer more information to learn from.


In [ ]:
# Cell 18 - Dataset Coverage Summary

from html import escape

from IPython.display import HTML, Markdown, display
import pandas as pd
import plotly.graph_objects as go


if "prices" not in globals():
    raise RuntimeError("Cell 18 needs the `prices` DataFrame from Cell 16. Rerun Cell 16, then rerun this cell.")

if "schema_validation" in globals():
    schema_status_column = "Status" if "Status" in schema_validation.columns else "status"
    if schema_status_column not in schema_validation.columns:
        raise RuntimeError("Cell 17 validation results do not include a status column. Rerun Cell 17, then rerun this cell.")
    failed_schema_checks = schema_validation.loc[
        schema_validation[schema_status_column].astype(str).str.lower().ne("pass")
    ]
    if not failed_schema_checks.empty:
        raise RuntimeError("Cell 17 found schema issues. Fix those before summarizing dataset coverage.")

required_columns = {"date", "symbol", "close"}
missing_columns = sorted(required_columns.difference(prices.columns))
if missing_columns:
    raise RuntimeError(
        "Cell 18 expected these columns in `prices`: "
        f"{', '.join(sorted(required_columns))}. Missing: {', '.join(missing_columns)}"
    )


coverage_prices = prices.loc[:, ["date", "symbol", "close"]].copy()
coverage_prices["date"] = pd.to_datetime(coverage_prices["date"], errors="coerce")
coverage_prices["symbol"] = coverage_prices["symbol"].astype(str).str.strip()

invalid_rows = coverage_prices["date"].isna() | coverage_prices["symbol"].eq("")
coverage_prices = coverage_prices.loc[~invalid_rows].copy()

if coverage_prices.empty:
    raise RuntimeError("No usable rows remained after cleaning dates and symbols.")

overall_start = coverage_prices["date"].min()
overall_end = coverage_prices["date"].max()
available_trading_dates = int(coverage_prices["date"].nunique())

symbol_coverage = (
    coverage_prices.groupby("symbol", as_index=False)
    .agg(
        observations=("close", "count"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        distinct_trading_dates=("date", "nunique"),
    )
    .sort_values(["observations", "symbol"], ascending=[False, True])
    .reset_index(drop=True)
)

symbol_coverage["calendar_span_days"] = (
    symbol_coverage["last_date"] - symbol_coverage["first_date"]
).dt.days + 1
symbol_coverage["coverage_years"] = (symbol_coverage["calendar_span_days"] / 365.25).round(2)
symbol_coverage["share_of_dataset_dates"] = (
    symbol_coverage["distinct_trading_dates"] / available_trading_dates
).round(3)
symbol_coverage["coverage_rank"] = range(1, len(symbol_coverage) + 1)

dataset_coverage_summary = pd.DataFrame(
    [
        {"Metric": "Price rows", "Value": f"{len(coverage_prices):,}"},
        {"Metric": "ETF symbols", "Value": f"{symbol_coverage['symbol'].nunique():,}"},
        {"Metric": "Trading dates", "Value": f"{available_trading_dates:,}"},
        {"Metric": "Earliest date", "Value": overall_start.strftime("%Y-%m-%d")},
        {"Metric": "Latest date", "Value": overall_end.strftime("%Y-%m-%d")},
        {
            "Metric": "Median observations per ETF",
            "Value": f"{int(symbol_coverage['observations'].median()):,}",
        },
        {
            "Metric": "Shortest ETF history",
            "Value": f"{int(symbol_coverage['observations'].min()):,} observations",
        },
        {
            "Metric": "Longest ETF history",
            "Value": f"{int(symbol_coverage['observations'].max()):,} observations",
        },
    ]
)


def _html_table(df, max_rows=12):
    shown = df.head(max_rows).copy()
    header = "".join(f"<th>{escape(str(col))}</th>" for col in shown.columns)
    rows = []
    for row in shown.itertuples(index=False):
        cells = "".join(f"<td>{escape(str(value))}</td>" for value in row)
        rows.append(f"<tr>{cells}</tr>")
    return (
        "<div class='coverage-table-scroll'>"
        "<table class='coverage-table'>"
        f"<thead><tr>{header}</tr></thead>"
        f"<tbody>{''.join(rows)}</tbody>"
        "</table>"
        "</div>"
    )


summary_html = _html_table(dataset_coverage_summary)
top_preview = symbol_coverage.loc[
    :,
    ["coverage_rank", "symbol", "observations", "first_date", "last_date", "share_of_dataset_dates"],
].head(10)
top_preview = top_preview.assign(
    first_date=top_preview["first_date"].dt.strftime("%Y-%m-%d"),
    last_date=top_preview["last_date"].dt.strftime("%Y-%m-%d"),
    share_of_dataset_dates=(top_preview["share_of_dataset_dates"] * 100).round(1).astype(str) + "%",
).rename(
    columns={
        "coverage_rank": "Rank",
        "symbol": "Symbol",
        "observations": "Observations",
        "first_date": "First date",
        "last_date": "Last date",
        "share_of_dataset_dates": "Dataset date share",
    }
)

display(
    HTML(
        """
        <style>
          .coverage-wrap {font-family: Arial, sans-serif; line-height: 1.45;}
          .coverage-grid {display: grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); gap: 12px; margin: 8px 0 16px;}
          .coverage-card {border: 1px solid #d8dee9; border-radius: 8px; padding: 12px; background: #ffffff;}
          .coverage-label {font-size: 12px; color: #5f6b7a; margin-bottom: 4px; text-transform: uppercase; letter-spacing: .02em;}
          .coverage-value {font-size: 22px; font-weight: 700; color: #1f2937;}
          .coverage-note {color: #4b5563; margin: 8px 0 14px;}
          .coverage-table-scroll {overflow-x: auto; margin: 6px 0 16px;}
          .coverage-table {border-collapse: collapse; width: 100%; min-width: 680px; font-size: 14px; background: #ffffff;}
          .coverage-table th {
            text-align: left;
            border-bottom: 2px solid #1f2937;
            padding: 9px 10px;
            background: #1f2937;
            color: #ffffff;
            font-weight: 700;
            white-space: nowrap;
          }
          .coverage-table td {
            border-bottom: 1px solid #e5e7eb;
            padding: 9px 10px;
            vertical-align: top;
            color: #111827;
            background: #ffffff;
          }
          .coverage-table tbody tr:nth-child(even) td {background: #f9fafb;}
        </style>
        """
        f"""
        <div class="coverage-wrap">
          <h3>Cell 18 - Dataset Coverage Summary</h3>
          <p class="coverage-note">
            Coverage tells us how much history each ETF contributes before we calculate returns or ask an optimizer to choose weights.
          </p>
          <div class="coverage-grid">
            <div class="coverage-card">
              <div class="coverage-label">ETF symbols</div>
              <div class="coverage-value">{symbol_coverage['symbol'].nunique():,}</div>
            </div>
            <div class="coverage-card">
              <div class="coverage-label">Price rows</div>
              <div class="coverage-value">{len(coverage_prices):,}</div>
            </div>
            <div class="coverage-card">
              <div class="coverage-label">Date range</div>
              <div class="coverage-value" style="font-size:17px;">{overall_start:%Y-%m-%d} to {overall_end:%Y-%m-%d}</div>
            </div>
            <div class="coverage-card">
              <div class="coverage-label">Median observations</div>
              <div class="coverage-value">{int(symbol_coverage['observations'].median()):,}</div>
            </div>
          </div>
          <h4>Dataset-wide coverage numbers</h4>
          {summary_html}
          <h4>Top symbols by available observations</h4>
          {_html_table(top_preview, max_rows=10)}
        </div>
        """
    )
)

top_n = min(30, len(symbol_coverage))
plot_data = symbol_coverage.head(top_n).sort_values("observations", ascending=True)

bar_color = globals().get("CHART_COLORS", {}).get("blue", "#2563eb")
accent_color = globals().get("CHART_COLORS", {}).get("teal", "#0f766e")

fig = go.Figure(
    go.Bar(
        x=plot_data["observations"],
        y=plot_data["symbol"],
        orientation="h",
        marker=dict(color=bar_color),
        customdata=plot_data[
            ["first_date", "last_date", "share_of_dataset_dates", "coverage_years"]
        ].to_numpy(),
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Observations: %{x:,}<br>"
            "First date: %{customdata[0]|%Y-%m-%d}<br>"
            "Last date: %{customdata[1]|%Y-%m-%d}<br>"
            "Share of dataset dates: %{customdata[2]:.1%}<br>"
            "Calendar span: %{customdata[3]:.2f} years"
            "<extra></extra>"
        ),
    )
)

fig.add_vline(
    x=float(symbol_coverage["observations"].median()),
    line_dash="dot",
    line_color=accent_color,
    annotation_text="median",
    annotation_position="top right",
)

fig.update_layout(
    title=f"Top {top_n} ETFs By Available Price Observations",
    xaxis_title="Price observations",
    yaxis_title="ETF symbol",
    height=max(520, 28 * top_n),
    margin=dict(l=80, r=30, t=70, b=60),
)

if "apply_theme" in globals():
    fig = apply_theme(fig, title=f"Top {top_n} ETFs By Available Price Observations", height=max(520, 28 * top_n))

display(fig)

display(
    Markdown(
        "**Cell 18 complete:** dataset coverage has been summarized. "
        "The next cell will explain why history length matters before optimization."
    )
)


## Cell 19 - What To Notice: Coverage

Cell 18 showed that not every ETF necessarily brings the same amount of usable history. That matters because portfolio research is only as good as the data window we give it.

When an ETF has more price observations, we usually have more information for estimating returns, volatility, drawdowns, and relationships with other ETFs. When an ETF has less history, the optimizer may still accept it, but the estimate behind that asset is based on a shorter record.

For this workshop, coverage is not a scorecard. A longer history does not automatically make an ETF better, and a shorter history does not automatically make it unusable. Coverage is a context check: it helps us understand how much evidence sits behind each asset before we ask skfolio to build portfolios.

What to notice before moving on:

- the dataset has enough shared history to support a teaching workflow
- symbols with shorter histories may need extra caution later
- the next step adds metadata and liquidity so we can inspect not just how much history each ETF has, but what each ETF represents and how tradable it appears


## Cell 20 - Metadata And Liquidity Join

Cell 18 told us how much price history each ETF has. Now we add context about what each ETF looks like as an instrument and how actively it trades.

Before you run the code cell below, here is what it will do:

- start from the covered ETF symbols in `symbol_coverage`
- join ETF metadata from `metadata`
- join recent trading-activity fields from `liquidity`
- create `etf_universe`, a one-row-per-symbol table we will reuse later
- show a compact preview with readable labels instead of raw column names

**Runtime note:** this cell expects Cell 16 through Cell 18 to have run successfully. If the runtime restarted, rerun Cell 14 through Cell 18 first.

**What to check after it runs:** every ETF from the price dataset should still be represented, and most rows should have liquidity data attached. Missing metadata or liquidity does not automatically break the workshop, but it is something we should notice before filtering the universe.


In [ ]:
# Cell 20 - Metadata And Liquidity Join

from html import escape

from IPython.display import HTML, Markdown, display
import pandas as pd


required_names = ["metadata", "liquidity", "symbol_coverage"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    missing_text = ", ".join(missing_names)
    raise RuntimeError(
        "Cell 20 needs the loaded data and coverage table first. "
        f"Run Cell 16 through Cell 18, then rerun this cell. Missing: {missing_text}"
    )

for frame_name in required_names:
    frame = globals()[frame_name]
    if not isinstance(frame, pd.DataFrame):
        raise TypeError(f"`{frame_name}` should be a pandas DataFrame, but found {type(frame).__name__}.")
    if "symbol" not in frame.columns:
        raise RuntimeError(f"`{frame_name}` needs a `symbol` column before Cell 20 can join the universe tables.")


def clean_symbol_column(frame):
    cleaned = frame.copy()
    cleaned["symbol"] = cleaned["symbol"].astype(str).str.strip().str.upper()
    cleaned = cleaned.loc[cleaned["symbol"].ne("")]
    return cleaned


coverage_base = clean_symbol_column(symbol_coverage).drop_duplicates("symbol").copy()
metadata_clean = clean_symbol_column(metadata).drop_duplicates("symbol").copy()
liquidity_clean = clean_symbol_column(liquidity).drop_duplicates("symbol").copy()

coverage_columns = [
    column
    for column in [
        "symbol",
        "observations",
        "first_date",
        "last_date",
        "distinct_trading_dates",
        "share_of_dataset_dates",
        "coverage_rank",
    ]
    if column in coverage_base.columns
]

metadata_columns = [
    column
    for column in [
        "symbol",
        "asset_type",
        "name",
        "type",
        "exchange",
        "exchange_name",
        "trading",
        "fractional_trading",
        "option_trading",
        "shorting_availability",
    ]
    if column in metadata_clean.columns
]

liquidity_columns = [
    column
    for column in [
        "symbol",
        "rank_avg_daily_volume",
        "rank_avg_daily_dollar_volume",
        "avg_daily_volume",
        "median_daily_volume",
        "avg_daily_dollar_volume",
        "median_daily_dollar_volume",
        "latest_volume",
        "latest_close",
        "lookback_bars",
        "lookback_days",
        "history_bars",
        "history_start",
        "history_end",
        "latest_date",
    ]
    if column in liquidity_clean.columns
]

etf_universe = (
    coverage_base.loc[:, coverage_columns]
    .merge(metadata_clean.loc[:, metadata_columns], on="symbol", how="left", validate="one_to_one")
    .merge(liquidity_clean.loc[:, liquidity_columns], on="symbol", how="left", validate="one_to_one")
)

metadata_check_columns = [column for column in metadata_columns if column != "symbol"]
liquidity_check_columns = [column for column in liquidity_columns if column != "symbol"]
etf_universe["has_metadata"] = (
    etf_universe[metadata_check_columns].notna().any(axis=1) if metadata_check_columns else False
)
etf_universe["has_liquidity"] = (
    etf_universe[liquidity_check_columns].notna().any(axis=1) if liquidity_check_columns else False
)

if "avg_daily_dollar_volume" in etf_universe.columns:
    etf_universe["avg_daily_dollar_volume_millions"] = etf_universe["avg_daily_dollar_volume"] / 1_000_000

if "share_of_dataset_dates" in etf_universe.columns:
    etf_universe["dataset_date_share_pct"] = etf_universe["share_of_dataset_dates"] * 100

universe_join_summary = pd.DataFrame(
    [
        {"Metric": "Symbols from price history", "Value": f"{len(coverage_base):,}"},
        {"Metric": "Rows in joined universe", "Value": f"{len(etf_universe):,}"},
        {"Metric": "Symbols with metadata", "Value": f"{int(etf_universe['has_metadata'].sum()):,}"},
        {"Metric": "Symbols with liquidity data", "Value": f"{int(etf_universe['has_liquidity'].sum()):,}"},
    ]
)


def format_number(value, decimals=0, suffix=""):
    if pd.isna(value):
        return "Missing"
    return f"{float(value):,.{decimals}f}{suffix}"


def format_status_value(value):
    if pd.isna(value):
        return "Missing"
    text = str(value).strip()
    if not text:
        return "Missing"
    return text.replace("_", " ").title()


preview_columns = [
    column
    for column in [
        "symbol",
        "exchange_name",
        "trading",
        "fractional_trading",
        "observations",
        "dataset_date_share_pct",
        "avg_daily_volume",
        "avg_daily_dollar_volume_millions",
        "rank_avg_daily_dollar_volume",
    ]
    if column in etf_universe.columns
]

universe_preview = etf_universe.sort_values(
    [column for column in ["rank_avg_daily_dollar_volume", "symbol"] if column in etf_universe.columns],
    ascending=True,
    na_position="last",
).loc[:, preview_columns].head(12)

display_preview = universe_preview.copy()
rename_map = {
    "symbol": "Symbol",
    "exchange_name": "Exchange",
    "trading": "Trading",
    "fractional_trading": "Fractional trading",
    "observations": "Price observations",
    "dataset_date_share_pct": "Dataset date share",
    "avg_daily_volume": "Average daily shares",
    "avg_daily_dollar_volume_millions": "Average daily dollars",
    "rank_avg_daily_dollar_volume": "Liquidity rank",
}

if "dataset_date_share_pct" in display_preview.columns:
    display_preview["dataset_date_share_pct"] = display_preview["dataset_date_share_pct"].map(
        lambda value: format_number(value, decimals=1, suffix="%")
    )
for status_column in ["trading", "fractional_trading"]:
    if status_column in display_preview.columns:
        display_preview[status_column] = display_preview[status_column].map(format_status_value)
if "avg_daily_volume" in display_preview.columns:
    display_preview["avg_daily_volume"] = display_preview["avg_daily_volume"].map(
        lambda value: format_number(value, decimals=0)
    )
if "avg_daily_dollar_volume_millions" in display_preview.columns:
    display_preview["avg_daily_dollar_volume_millions"] = display_preview[
        "avg_daily_dollar_volume_millions"
    ].map(lambda value: format_number(value, decimals=1, suffix="M"))
if "rank_avg_daily_dollar_volume" in display_preview.columns:
    display_preview["rank_avg_daily_dollar_volume"] = display_preview[
        "rank_avg_daily_dollar_volume"
    ].map(lambda value: "Missing" if pd.isna(value) else f"{int(value):,}")

display_preview = display_preview.rename(columns=rename_map)


def html_table(frame, table_class="universe-table"):
    header = "".join(f"<th>{escape(str(column))}</th>" for column in frame.columns)
    rows = []
    for row in frame.itertuples(index=False):
        cells = "".join(f"<td>{escape(str(value))}</td>" for value in row)
        rows.append(f"<tr>{cells}</tr>")
    return (
        f"<div class='{table_class}-scroll'>"
        f"<table class='{table_class}'>"
        f"<thead><tr>{header}</tr></thead>"
        f"<tbody>{''.join(rows)}</tbody>"
        "</table>"
        "</div>"
    )


summary_html = html_table(universe_join_summary)
preview_html = html_table(display_preview)

display(
    HTML(
        """
        <style>
          .universe-wrap {font-family: Arial, sans-serif; line-height: 1.45;}
          .universe-note {color: #4b5563; margin: 8px 0 14px;}
          .universe-table-scroll {overflow-x: auto; margin: 6px 0 16px;}
          .universe-table {border-collapse: collapse; width: 100%; min-width: 760px; font-size: 14px; background: #ffffff;}
          .universe-table th {
            text-align: left;
            border-bottom: 2px solid #1f2937;
            padding: 9px 10px;
            background: #1f2937;
            color: #ffffff;
            font-weight: 700;
            white-space: nowrap;
          }
          .universe-table td {
            border-bottom: 1px solid #e5e7eb;
            padding: 9px 10px;
            vertical-align: top;
            color: #111827;
            background: #ffffff;
          }
          .universe-table tbody tr:nth-child(even) td {background: #f9fafb;}
        </style>
        """
        f"""
        <div class="universe-wrap">
          <h3>Cell 20 - Metadata And Liquidity Join</h3>
          <p class="universe-note">
            This joined table gives each ETF a richer profile: coverage, tradeability fields, and recent liquidity measures.
          </p>
          <h4>Join checks</h4>
          {summary_html}
          <h4>Preview of the joined ETF universe</h4>
          {preview_html}
        </div>
        """
    )
)

missing_liquidity_symbols = sorted(etf_universe.loc[~etf_universe["has_liquidity"], "symbol"].tolist())
if missing_liquidity_symbols:
    display(
        Markdown(
            "**Liquidity note:** these symbols do not have liquidity rows yet: "
            + ", ".join(f"`{symbol}`" for symbol in missing_liquidity_symbols[:12])
            + ("..." if len(missing_liquidity_symbols) > 12 else "")
        )
    )

display(
    Markdown(
        "**Cell 20 complete:** `etf_universe` now combines coverage, metadata, and liquidity. "
        "The next cell ranks ETFs by recent trading activity."
    )
)


## Cell 21 - Liquidity Ranking Charts

Liquidity tells us how actively an ETF has traded recently. It does not tell us which ETF will perform best, but it helps us avoid building examples around instruments that may be hard to trade or expensive to enter and exit.

Before you run the code cell below, here is what it will do:

- use `etf_universe` from Cell 20
- rank ETFs by average daily dollar volume
- rank ETFs by average daily share volume
- create `liquidity_rankings`, a reusable sorted table
- show two horizontal bar charts that are easier to scan than a wide raw table

**Runtime note:** this cell expects Cell 20 to have created `etf_universe`. If the runtime restarted, rerun Cell 14 through Cell 20 first.

**What to check after it runs:** the highest-liquidity ETFs should appear at the top of each chart, and the chart labels should be readable without needing to scroll sideways.


In [ ]:
# Cell 21 - Liquidity Ranking Charts

from IPython.display import Markdown, display
import pandas as pd
import plotly.graph_objects as go


if "etf_universe" not in globals():
    raise RuntimeError("Cell 21 needs `etf_universe` from Cell 20. Run Cell 20, then rerun this cell.")

required_liquidity_columns = ["symbol", "avg_daily_volume", "avg_daily_dollar_volume"]
missing_liquidity_columns = [
    column for column in required_liquidity_columns if column not in etf_universe.columns
]
if missing_liquidity_columns:
    raise RuntimeError(
        "Cell 21 expected liquidity columns from Cell 20. Missing: "
        + ", ".join(missing_liquidity_columns)
    )

liquidity_rankings = etf_universe.loc[
    :,
    [
        column
        for column in [
            "symbol",
            "exchange_name",
            "avg_daily_volume",
            "median_daily_volume",
            "avg_daily_dollar_volume",
            "median_daily_dollar_volume",
            "rank_avg_daily_volume",
            "rank_avg_daily_dollar_volume",
            "latest_close",
            "lookback_days",
            "history_start",
            "history_end",
        ]
        if column in etf_universe.columns
    ],
].copy()

liquidity_rankings = liquidity_rankings.dropna(
    subset=["avg_daily_volume", "avg_daily_dollar_volume"],
    how="any",
).copy()

if liquidity_rankings.empty:
    raise RuntimeError("No ETFs with both share-volume and dollar-volume liquidity data were available.")

liquidity_rankings["avg_daily_dollar_volume_millions"] = (
    liquidity_rankings["avg_daily_dollar_volume"] / 1_000_000
)
liquidity_rankings["avg_daily_volume_millions"] = liquidity_rankings["avg_daily_volume"] / 1_000_000

liquidity_rankings = liquidity_rankings.sort_values(
    ["avg_daily_dollar_volume", "symbol"],
    ascending=[False, True],
).reset_index(drop=True)
liquidity_rankings["dollar_volume_rank"] = range(1, len(liquidity_rankings) + 1)
liquidity_rankings["share_volume_rank"] = (
    liquidity_rankings["avg_daily_volume"].rank(method="min", ascending=False).astype(int)
)

top_n = min(20, len(liquidity_rankings))
top_dollar = liquidity_rankings.nlargest(top_n, "avg_daily_dollar_volume").sort_values(
    "avg_daily_dollar_volume"
)
top_shares = liquidity_rankings.nlargest(top_n, "avg_daily_volume").sort_values(
    "avg_daily_volume"
)

bar_blue = globals().get("CHART_COLORS", {}).get("blue", "#2563eb")
bar_teal = globals().get("CHART_COLORS", {}).get("teal", "#0f766e")


def horizontal_bar_chart(data, value_column, title, xaxis_title, color):
    chart_title = dict(
        text=title,
        x=0.5,
        xanchor="center",
        y=0.96,
        yanchor="top",
        font=dict(size=18),
    )
    fig = go.Figure(
        go.Bar(
            x=data[value_column],
            y=data["symbol"],
            orientation="h",
            marker=dict(color=color),
            text=data[value_column].map(lambda value: f"{value:,.1f}"),
            textposition="outside",
            cliponaxis=False,
            hovertemplate="<b>%{y}</b><br>" + xaxis_title + ": %{x:,.2f}<extra></extra>",
        )
    )
    fig.update_layout(
        title=chart_title,
        xaxis_title=xaxis_title,
        yaxis_title="ETF symbol",
        height=max(480, 24 * len(data) + 180),
        margin=dict(l=80, r=110, t=70, b=60),
    )
    if "apply_theme" in globals():
        fig = apply_theme(fig, title=title, height=max(480, 24 * len(data) + 180))
        fig.update_layout(
            title=chart_title,
            margin=dict(l=80, r=110, t=70, b=60),
        )
    return fig


display(
    Markdown(
        "### Cell 21 - Liquidity Ranking Charts\n\n"
        "The first chart ranks ETFs by average daily dollar volume, which combines trading activity with price. "
        "The second chart ranks ETFs by average daily share volume, which shows how many shares traded on average. "
        "Neither ranking is a performance forecast; both are practical market-structure checks."
    )
)

display(
    horizontal_bar_chart(
        top_dollar,
        "avg_daily_dollar_volume_millions",
        f"Top {top_n} ETFs By Average Daily Dollar Volume",
        "Average daily dollar volume, millions",
        bar_blue,
    )
)

display(
    horizontal_bar_chart(
        top_shares,
        "avg_daily_volume_millions",
        f"Top {top_n} ETFs By Average Daily Share Volume",
        "Average daily share volume, millions",
        bar_teal,
    )
)

top_dollar_symbol = liquidity_rankings.iloc[0]["symbol"]
top_dollar_value = liquidity_rankings.iloc[0]["avg_daily_dollar_volume_millions"]
median_dollar_value = liquidity_rankings["avg_daily_dollar_volume_millions"].median()

display(
    Markdown(
        f"**Quick read:** `{top_dollar_symbol}` has the highest average daily dollar volume in this dataset "
        f"at about USD {top_dollar_value:,.1f} million. The median ETF in the liquidity table is about "
        f"USD {median_dollar_value:,.1f} million. The next section turns those ETF inputs into a return matrix."
    )
)

display(
    Markdown(
        "**Cell 21 complete:** liquidity rankings are ready. "
        "Later, we can use these fields to keep the optimizer focused on a practical ETF universe."
    )
)


## Cell 22 - Prices Are Not Returns

Before we build portfolios, we need to change the shape of the problem. ETF prices are useful raw history, but prices by themselves are not comparable. A USD 500 ETF is not automatically more expensive or more important than a USD 50 ETF. What matters for portfolio research is how each asset changed through time.

That is why the next section turns prices into returns. A return measures percentage change:

`simple return = today's price / yesterday's price - 1`

This notebook will use simple, or linear, returns because they are intuitive and connect directly to portfolio weights. If an ETF has a 2% weight and earns a 1% daily return, its contribution to the portfolio that day is easy to reason about.

You may also see log returns in quantitative finance:

`log return = log(today's price / yesterday's price)`

Log returns have useful mathematical properties, especially when chaining returns over time, but simple returns are the clearer convention for this workshop unless we intentionally compare the two.

In skfolio language, the modeling input is usually called `X`: a returns matrix where each row is a date, each column is an asset, and each value is that asset's return for that date. The next code cells will:

- pivot the long price table into a date-by-symbol price matrix
- check for missing prices
- convert prices into returns
- keep the result in a reusable variable named `returns`

**What to remember:** prices tell us where each ETF traded. Returns tell us how each ETF moved. Portfolio optimization needs returns.


## Cell 23 - Pivot Prices Wide

Now we turn the long price table into the shape that portfolio tools expect.

The raw `prices` table has one row per ETF per date. That is useful for storage, but portfolio math is easier when each row is one date and each column is one ETF symbol. The next code cell will:

- clean the `date`, `symbol`, and `close` fields
- check whether any date-symbol pairs appear more than once
- collapse duplicate date-symbol pairs safely if they exist
- create `prices_wide`, a date-by-symbol close-price matrix
- show a small summary so the shape is easy to verify

**Runtime note:** this cell expects Cell 16 to have loaded `prices`. If the runtime restarted, rerun Cell 14 through Cell 16 first.

**What to check after it runs:** `prices_wide` should have dates down the rows, ETF symbols across the columns, and one close price per date-symbol pair.


In [ ]:
# Cell 23 - Pivot Prices Wide

from html import escape

from IPython.display import HTML, Markdown, display
import pandas as pd


if "prices" not in globals():
    raise RuntimeError("Cell 23 needs the `prices` DataFrame from Cell 16. Rerun Cell 16, then rerun this cell.")

required_columns = {"date", "symbol", "close"}
missing_columns = sorted(required_columns.difference(prices.columns))
if missing_columns:
    raise RuntimeError(
        "Cell 23 expected `prices` to include date, symbol, and close. "
        f"Missing: {', '.join(missing_columns)}"
    )

price_matrix_source = prices.loc[:, ["date", "symbol", "close"]].copy()
price_matrix_source["date"] = pd.to_datetime(price_matrix_source["date"], errors="coerce")
price_matrix_source["symbol"] = price_matrix_source["symbol"].astype(str).str.strip().str.upper()
price_matrix_source["close"] = pd.to_numeric(price_matrix_source["close"], errors="coerce")

invalid_rows = (
    price_matrix_source["date"].isna()
    | price_matrix_source["symbol"].eq("")
    | price_matrix_source["close"].isna()
    | price_matrix_source["close"].le(0)
)
clean_price_rows = price_matrix_source.loc[~invalid_rows].copy()

if clean_price_rows.empty:
    raise RuntimeError("No usable price rows remained after cleaning dates, symbols, and close prices.")

duplicate_pairs = int(clean_price_rows.duplicated(["date", "symbol"]).sum())
if duplicate_pairs:
    clean_price_rows = (
        clean_price_rows.sort_values(["symbol", "date"])
        .groupby(["date", "symbol"], as_index=False)
        .agg(close=("close", "last"))
    )

prices_wide = (
    clean_price_rows.pivot(index="date", columns="symbol", values="close")
    .sort_index()
    .sort_index(axis=1)
)
prices_wide.index.name = "date"
prices_wide.columns.name = "symbol"

if not prices_wide.index.is_monotonic_increasing:
    raise RuntimeError("`prices_wide` index should be sorted by date, but it is not.")

remaining_duplicate_pairs = int(clean_price_rows.duplicated(["date", "symbol"]).sum())
if remaining_duplicate_pairs:
    raise RuntimeError("Duplicate date-symbol pairs remain after pivot preparation.")

missing_cells = int(prices_wide.isna().sum().sum())
total_cells = int(prices_wide.shape[0] * prices_wide.shape[1])
missing_pct = missing_cells / total_cells if total_cells else 0

price_matrix_summary = pd.DataFrame(
    [
        {"Check": "Clean source rows", "Result": f"{len(clean_price_rows):,}"},
        {"Check": "ETF columns", "Result": f"{prices_wide.shape[1]:,}"},
        {"Check": "Trading-date rows", "Result": f"{prices_wide.shape[0]:,}"},
        {"Check": "Date range", "Result": f"{prices_wide.index.min().date()} to {prices_wide.index.max().date()}"},
        {"Check": "Duplicate date-symbol rows collapsed", "Result": f"{duplicate_pairs:,}"},
        {"Check": "Missing matrix cells", "Result": f"{missing_cells:,} ({missing_pct:.1%})"},
    ]
)

preview = prices_wide.iloc[:5, : min(6, prices_wide.shape[1])].round(2).reset_index()
preview["date"] = preview["date"].dt.strftime("%Y-%m-%d")


def render_table(frame, class_name="pivot-table"):
    header = "".join(f"<th>{escape(str(column))}</th>" for column in frame.columns)
    rows = []
    for row in frame.itertuples(index=False):
        cells = "".join(f"<td>{escape(str(value))}</td>" for value in row)
        rows.append(f"<tr>{cells}</tr>")
    return f"<div class='{class_name}-scroll'><table class='{class_name}'><thead><tr>{header}</tr></thead><tbody>{''.join(rows)}</tbody></table></div>"

summary_html = render_table(price_matrix_summary, "pivot-summary")
preview_html = render_table(preview, "pivot-preview")

display(
    HTML(
        f"""
        <style>
          .pivot-wrap {{font-family: Arial, sans-serif; line-height: 1.45; max-width: 1100px;}}
          .pivot-note {{color: #374151; margin: 6px 0 14px;}}
          .pivot-summary-scroll, .pivot-preview-scroll {{overflow-x: auto; margin: 8px 0 18px;}}
          .pivot-summary, .pivot-preview {{border-collapse: collapse; width: 100%; background: #ffffff; font-size: 14px;}}
          .pivot-summary th, .pivot-preview th {{background: #111827; color: #ffffff; text-align: left; padding: 9px 10px; white-space: nowrap;}}
          .pivot-summary td, .pivot-preview td {{border-bottom: 1px solid #e5e7eb; color: #111827; padding: 9px 10px; background: #ffffff;}}
          .pivot-summary tbody tr:nth-child(even) td, .pivot-preview tbody tr:nth-child(even) td {{background: #f9fafb;}}
        </style>
        <div class="pivot-wrap">
          <h3>Cell 23 - Pivot Prices Wide</h3>
          <p class="pivot-note">The price table is now a matrix: dates are rows, ETF symbols are columns, and values are close prices.</p>
          {summary_html}
          <h4>First rows and first symbols</h4>
          {preview_html}
          <p class="pivot-note"><b>Cell 23 complete:</b> <code>prices_wide</code> is ready for missingness checks.</p>
        </div>
        """
    )
)


## Cell 24 - Missingness Heatmap

Before we calculate returns, we need to see where price data is missing.

Missing prices are normal in ETF universes because funds can launch at different times, trade on different calendars, or have sparse histories. The next code cell will:

- measure missing close prices in `prices_wide`
- create `missingness_summary`, a per-symbol missingness table
- compress daily data into a monthly coverage heatmap so the visual stays readable
- highlight whether missingness is mostly early-inception history or a broader data-quality problem

**Runtime note:** this cell expects Cell 23 to have created `prices_wide`. If the runtime restarted, rerun Cell 14 through Cell 23 first.

**What to check after it runs:** green areas mean the ETF has prices for most dates in that month; lighter or warmer areas mean coverage is thinner. The chart is a diagnostic, not a performance ranking.


In [ ]:
# Cell 24 - Missingness Heatmap

from IPython.display import Markdown, display
import pandas as pd
import plotly.graph_objects as go


if "prices_wide" not in globals():
    raise RuntimeError("Cell 24 needs `prices_wide` from Cell 23. Run Cell 23, then rerun this cell.")

if not isinstance(prices_wide, pd.DataFrame) or prices_wide.empty:
    raise RuntimeError("`prices_wide` should be a non-empty pandas DataFrame.")

missingness_summary = pd.DataFrame(
    {
        "symbol": prices_wide.columns,
        "available_prices": prices_wide.notna().sum().to_numpy(),
        "missing_prices": prices_wide.isna().sum().to_numpy(),
        "missing_share": prices_wide.isna().mean().to_numpy(),
        "first_price_date": [prices_wide[column].first_valid_index() for column in prices_wide.columns],
        "last_price_date": [prices_wide[column].last_valid_index() for column in prices_wide.columns],
    }
)
missingness_summary["available_share"] = 1 - missingness_summary["missing_share"]
missingness_summary = missingness_summary.sort_values(
    ["available_share", "symbol"], ascending=[False, True]
).reset_index(drop=True)
missingness_summary["coverage_rank"] = range(1, len(missingness_summary) + 1)

selected_symbol_count = min(40, len(missingness_summary))
heatmap_symbols = missingness_summary.head(selected_symbol_count)["symbol"].tolist()

monthly_presence = prices_wide[heatmap_symbols].notna().astype(float).resample("ME").mean()
monthly_price_coverage = monthly_presence.T
monthly_price_coverage.index.name = "symbol"
monthly_price_coverage.columns = monthly_price_coverage.columns.strftime("%Y-%m")

coverage_values = monthly_price_coverage.to_numpy()

fig = go.Figure(
    data=go.Heatmap(
        z=coverage_values,
        x=monthly_price_coverage.columns,
        y=monthly_price_coverage.index,
        zmin=0,
        zmax=1,
        colorscale=[
            [0.00, "#b91c1c"],
            [0.50, "#f59e0b"],
            [0.85, "#fde68a"],
            [1.00, "#0f766e"],
        ],
        colorbar=dict(title="Monthly<br>coverage", tickformat=".0%"),
        hovertemplate="<b>%{y}</b><br>%{x}<br>Coverage: %{z:.0%}<extra></extra>",
    )
)
chart_height = max(520, 22 * selected_symbol_count + 180)
chart_title = dict(
    text=f"Monthly Price Coverage For The {selected_symbol_count} Most Complete ETFs",
    x=0.5,
    xanchor="center",
    y=0.97,
    yanchor="top",
    font=dict(size=18),
)
fig.update_layout(
    title=chart_title,
    xaxis_title="Month",
    yaxis_title="ETF symbol",
    height=chart_height,
    margin=dict(l=90, r=90, t=80, b=80),
)
if "apply_theme" in globals():
    fig = apply_theme(fig, title=chart_title, height=chart_height)
    fig.update_layout(margin=dict(l=90, r=90, t=80, b=80))

coverage_checks = pd.DataFrame(
    [
        {"Metric": "ETF symbols checked", "Value": f"{len(missingness_summary):,}"},
        {"Metric": "Symbols shown in heatmap", "Value": f"{selected_symbol_count:,}"},
        {"Metric": "Median available share", "Value": f"{missingness_summary['available_share'].median():.1%}"},
        {"Metric": "Lowest available share", "Value": f"{missingness_summary['available_share'].min():.1%}"},
        {"Metric": "Highest available share", "Value": f"{missingness_summary['available_share'].max():.1%}"},
    ]
)

if "show_metric_table" in globals():
    show_metric_table(coverage_checks, title="Missingness Summary", height=310).show()
else:
    display(coverage_checks)

fig.show()

display(
    Markdown(
        "**Cell 24 complete:** `missingness_summary` and `monthly_price_coverage` are ready. "
        "The next cell will convert the price matrix into returns while keeping the default path simple and linear."
    )
)


## Cell 25 - Convert Prices To Returns

This is the key transformation in the first notebook: prices become returns.

Portfolio models do not compare raw price levels. They compare percentage changes through time. The next code cell will:

- use `skfolio.preprocessing.prices_to_returns` on `prices_wide`
- create the reusable return matrix named `returns`
- show the equivalent pandas simple-return calculation once for learning
- confirm the return matrix shape, date range, and missing-value profile
- make the row-count change clear, because later-starting ETFs can shorten the shared return window

**Runtime note:** this cell expects Cell 23 to have created `prices_wide`. Cell 24 is useful for diagnostics, but Cell 25 only needs the price matrix.

**What to check after it runs:** `returns` should have dates as rows, ETF symbols as columns, and daily percentage returns as values. These returns become the main input for risk, correlation, validation, and optimization later.


In [ ]:
# Cell 25 - Convert Prices To Returns

from IPython.display import Markdown, display
import numpy as np
import pandas as pd

from skfolio.preprocessing import prices_to_returns


if "prices_wide" not in globals():
    raise RuntimeError("Cell 25 needs `prices_wide` from Cell 23. Run Cell 23, then rerun this cell.")

if not isinstance(prices_wide, pd.DataFrame) or prices_wide.empty:
    raise RuntimeError("`prices_wide` should be a non-empty pandas DataFrame before converting prices to returns.")

price_rows_before_conversion = prices_wide.shape[0]
price_start = prices_wide.index.min()
price_end = prices_wide.index.max()
raw_simple_returns = prices_wide.pct_change(fill_method=None).iloc[1:]

returns = prices_to_returns(prices_wide, log_returns=False)
returns = returns.sort_index().sort_index(axis=1)
returns.index.name = "date"
returns.columns.name = "symbol"

pandas_simple_returns = raw_simple_returns.reindex(index=returns.index, columns=returns.columns)
comparison_difference = (returns - pandas_simple_returns).abs().max(skipna=True).max(skipna=True)
if pd.isna(comparison_difference):
    comparison_label = "not comparable because all aligned differences are missing"
else:
    comparison_label = f"{comparison_difference:.2e} max absolute difference"

rows_not_retained = raw_simple_returns.shape[0] - returns.shape[0]
if rows_not_retained > 0:
    retained_note = (
        f"{rows_not_retained:,} early rows not retained so the full matrix has usable returns for every ETF"
    )
else:
    retained_note = "No return rows were removed after conversion"

return_summary = pd.DataFrame(
    [
        {"Check": "Return convention", "Result": "Linear/simple returns"},
        {"Check": "Created object", "Result": "returns"},
        {"Check": "Price rows before conversion", "Result": f"{price_rows_before_conversion:,}"},
        {"Check": "Price date range", "Result": f"{price_start.date()} to {price_end.date()}"},
        {"Check": "Return rows", "Result": f"{returns.shape[0]:,}"},
        {"Check": "ETF columns", "Result": f"{returns.shape[1]:,}"},
        {"Check": "Date range", "Result": f"{returns.index.min().date()} to {returns.index.max().date()}"},
        {"Check": "Missing return cells", "Result": f"{int(returns.isna().sum().sum()):,}"},
        {"Check": "Shared-window note", "Result": retained_note},
        {"Check": "Pandas formula check", "Result": comparison_label},
    ]
)

if "show_metric_table" in globals():
    show_metric_table(return_summary, title="Return Matrix Summary", height=360).show()
else:
    display(return_summary)

preview = returns.iloc[:5, : min(6, returns.shape[1])].mul(100).round(3)
display(Markdown("### First rows of `returns` shown as daily percentages"))
display(preview)

display(
    Markdown(
        "**Cell 25 complete:** `returns` is now the main modeling table for the rest of the masterclass. "
        "We used linear returns because portfolio returns are weighted averages of asset returns. "
        "The return window may start later than the raw price window when the full ETF universe needs a shared, complete history. "
        "Later roadmap sections can compare log returns and other return transformations deliberately."
    )
)


## Cell 26 - What To Notice: Return Matrix

Cell 25 created the most important input table in the notebook: `returns`.

A return matrix is easy to overlook because it looks like just another table. It is more than that. In skfolio-style portfolio research, this is the table most estimators learn from:

- each row is one observation date
- each column is one ETF
- each value is the ETF's return for that date

The reason we use simple, or linear, returns here is practical. Portfolio weights combine asset returns across the same date. If a portfolio has 60% in one ETF and 40% in another, the portfolio's daily linear return is the weighted average of those two ETF returns.

Log returns are still useful to know. They chain neatly through time and appear often in quantitative finance. But for the main optimizer path in this masterclass, linear returns keep the math closer to how portfolio weights are applied.

What to remember before moving on:

- prices describe levels
- returns describe movement
- missing data affects which assets can be compared fairly
- `returns` is the bridge from raw market history to risk, correlation, validation, and optimization

The next two cells make this more visual: first by normalizing prices to a common starting value, then by comparing the shape of daily return distributions.


## Cell 27 - Normalized Price Lines

Now that we have returns, we can still use prices for intuition.

Raw ETF prices start at different levels, so plotting them directly can be misleading. A normalized price chart resets each selected ETF to 100 at its first available price. That makes the lines easier to compare visually. The next code cell will:

- choose a small set of liquid, well-covered ETFs when possible
- normalize each selected price series to 100
- create `normalized_prices`
- show an interactive line chart with a centered title

**Runtime note:** this cell expects `prices_wide` from Cell 23 and works best if Cell 21 created `liquidity_rankings`.

**What to check after it runs:** the chart should compare paths, not raw dollar prices. A higher line means stronger cumulative movement over the displayed window, not a more expensive ETF.


In [ ]:
# Cell 27 - Normalized Price Lines

from IPython.display import Markdown, display
import pandas as pd
import plotly.graph_objects as go


if "prices_wide" not in globals():
    raise RuntimeError("Cell 27 needs `prices_wide` from Cell 23. Run Cell 23, then rerun this cell.")

if not isinstance(prices_wide, pd.DataFrame) or prices_wide.empty:
    raise RuntimeError("`prices_wide` should be a non-empty pandas DataFrame.")

if "liquidity_rankings" in globals() and "symbol" in liquidity_rankings.columns:
    candidate_symbols = liquidity_rankings.sort_values("avg_daily_dollar_volume", ascending=False)["symbol"].tolist()
elif "missingness_summary" in globals() and "symbol" in missingness_summary.columns:
    candidate_symbols = missingness_summary.sort_values("available_share", ascending=False)["symbol"].tolist()
else:
    candidate_symbols = prices_wide.notna().sum().sort_values(ascending=False).index.tolist()

normalized_symbols = [symbol for symbol in candidate_symbols if symbol in prices_wide.columns][:8]
if not normalized_symbols:
    raise RuntimeError("No symbols were available to normalize.")

normalized_prices = pd.DataFrame(index=prices_wide.index)
for symbol in normalized_symbols:
    series = prices_wide[symbol].dropna()
    if series.empty:
        continue
    normalized_prices[symbol] = prices_wide[symbol] / series.iloc[0] * 100

normalized_prices = normalized_prices.dropna(how="all")
if normalized_prices.empty:
    raise RuntimeError("No normalized price values were available after selecting symbols.")

fig = go.Figure()
for symbol in normalized_prices.columns:
    fig.add_trace(
        go.Scatter(
            x=normalized_prices.index,
            y=normalized_prices[symbol],
            mode="lines",
            name=symbol,
            hovertemplate=f"<b>{symbol}</b><br>%{{x|%Y-%m-%d}}<br>Index: %{{y:,.1f}}<extra></extra>",
        )
    )

chart_title = dict(
    text="Normalized Price Paths For Selected ETFs",
    x=0.5,
    xanchor="center",
    y=0.96,
    yanchor="top",
    font=dict(size=18),
)
fig.update_layout(
    title=chart_title,
    xaxis_title="Date",
    yaxis_title="Normalized price, first available value = 100",
    height=560,
    margin=dict(l=70, r=40, t=80, b=70),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
)
if "apply_theme" in globals():
    fig = apply_theme(fig, title=chart_title, height=560)
    fig.update_layout(
        margin=dict(l=70, r=40, t=80, b=70),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
    )
fig.add_hline(y=100, line_dash="dot", line_color="#6b7280")
fig.show()

normalized_price_summary = pd.DataFrame(
    {
        "Symbol": normalized_prices.columns,
        "First normalized value": [f"{normalized_prices[col].dropna().iloc[0]:.1f}" for col in normalized_prices.columns],
        "Last normalized value": [f"{normalized_prices[col].dropna().iloc[-1]:.1f}" for col in normalized_prices.columns],
        "Available points": [f"{normalized_prices[col].notna().sum():,}" for col in normalized_prices.columns],
    }
)

if "show_metric_table" in globals():
    show_metric_table(normalized_price_summary, title="Normalized Price Check", height=360).show()
else:
    display(normalized_price_summary)

display(
    Markdown(
        "**Cell 27 complete:** `normalized_prices` shows selected ETF price paths on the same starting scale. "
        "The next cell switches back to `returns` and compares daily movement distributions."
    )
)


## Cell 28 - Return Distributions, Correlation Teaser, And Notebook 01 Wrap-Up

The final code cell of Notebook 01 looks at the shape of daily ETF returns, then leaves you with the question that drives Notebook 02: which assets might actually diversify each other?

Average return is only one part of the story. Two ETFs can have similar averages but very different daily behavior. The next code cell will:

- use the `returns` matrix from Cell 25
- select a readable group of ETFs, usually the same symbols used in Cell 27
- create `return_distribution_summary`
- plot daily return histograms so the spread and tails are easier to see
- add a finer near-zero return table so rounded chart labels do not mislead you
- create a correlation heatmap teaser for a broader ETF sample
- show a short glossary of the core objects created in Notebook 01
- preview how Notebook 02 will use `returns` for portfolio objects, risk, and diversification

**Important idea:** low correlation does not automatically mean "good portfolio." It means the assets have moved differently in this sample, which may create diversification potential. Notebook 02 will test that more seriously.

**Runtime note:** this cell expects Cell 25 to have created `returns`. It works best if Cell 27 also created `normalized_symbols`.

**What to check after it runs:** wider histograms mean more variable daily returns, and long left tails matter because large down days shape drawdown risk. If the tallest bar appears near 0%, use the near-zero detail table to separate exact 0.00% days from many small returns that are visually rounded toward zero. In the heatmap, cooler or lighter relationships can be a clue that assets are less tied together, but the next notebook is where we turn that clue into portfolio research.


In [ ]:
# Cell 28 - Return Distributions And Correlation Teaser

from html import escape

from IPython.display import HTML, Markdown, display
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


if "returns" not in globals():
    raise RuntimeError("Cell 28 needs `returns` from Cell 25. Run Cell 25, then rerun this cell.")

if not isinstance(returns, pd.DataFrame) or returns.empty:
    raise RuntimeError("`returns` should be a non-empty pandas DataFrame.")

if "normalized_symbols" in globals():
    candidate_symbols = [symbol for symbol in normalized_symbols if symbol in returns.columns]
elif "liquidity_rankings" in globals() and "symbol" in liquidity_rankings.columns:
    candidate_symbols = [symbol for symbol in liquidity_rankings["symbol"].tolist() if symbol in returns.columns]
else:
    candidate_symbols = returns.notna().sum().sort_values(ascending=False).index.tolist()

distribution_symbols = candidate_symbols[:6]
if not distribution_symbols:
    raise RuntimeError("No symbols were available for return distribution plots.")

return_distribution_summary = (
    returns[distribution_symbols]
    .agg(["count", "mean", "std", "min", "median", "max"])
    .T.reset_index()
    .rename(
        columns={
            "index": "Symbol",
            "symbol": "Symbol",
            "count": "Observations",
            "mean": "Mean daily return",
            "std": "Daily volatility",
            "min": "Worst daily return",
            "median": "Median daily return",
            "max": "Best daily return",
        }
    )
)
for column in ["Mean daily return", "Daily volatility", "Worst daily return", "Median daily return", "Best daily return"]:
    return_distribution_summary[column] = return_distribution_summary[column].map(lambda value: f"{value:.2%}")
return_distribution_summary["Observations"] = return_distribution_summary["Observations"].astype(int).map(lambda value: f"{value:,}")

near_zero_rows = []
fine_bin_width_pct = 0.10


def format_pct_edge(value):
    if abs(value) < 0.005:
        value = 0.0
    return f"{value:.2f}%"


for symbol in distribution_symbols:
    daily_returns_pct = returns[symbol].dropna() * 100
    if daily_returns_pct.empty:
        continue
    bin_start = np.floor(daily_returns_pct.min() / fine_bin_width_pct) * fine_bin_width_pct
    bin_end = np.ceil(daily_returns_pct.max() / fine_bin_width_pct) * fine_bin_width_pct + fine_bin_width_pct
    fine_bins = np.arange(bin_start, bin_end + fine_bin_width_pct, fine_bin_width_pct)
    fine_counts, fine_edges = np.histogram(daily_returns_pct, bins=fine_bins)
    densest_idx = int(fine_counts.argmax())
    densest_left = fine_edges[densest_idx]
    densest_right = fine_edges[densest_idx + 1]
    exact_zero_days = int((daily_returns_pct == 0).sum())
    near_zero_days = int(daily_returns_pct.between(-0.10, 0.10, inclusive="both").sum())
    near_zero_rows.append(
        {
            "Symbol": symbol,
            "Exact 0.00% days": f"{exact_zero_days:,} ({exact_zero_days / len(daily_returns_pct):.1%})",
            "Within +/-0.10%": f"{near_zero_days:,} ({near_zero_days / len(daily_returns_pct):.1%})",
            "Densest 0.10% band": f"{format_pct_edge(densest_left)} to {format_pct_edge(densest_right)}",
            "Band days": f"{int(fine_counts[densest_idx]):,}",
        }
    )

near_zero_return_detail = pd.DataFrame(near_zero_rows)

rows = int(np.ceil(len(distribution_symbols) / 2))
fig = make_subplots(
    rows=rows,
    cols=2,
    subplot_titles=distribution_symbols,
    horizontal_spacing=0.10,
    vertical_spacing=0.14,
)

bar_color = globals().get("CHART_COLORS", {}).get("blue", "#2563eb")
for idx, symbol in enumerate(distribution_symbols):
    row = idx // 2 + 1
    col = idx % 2 + 1
    daily_returns_pct = returns[symbol].dropna() * 100
    fig.add_trace(
        go.Histogram(
            x=daily_returns_pct,
            xbins=dict(size=0.25),
            marker_color=bar_color,
            opacity=0.82,
            hovertemplate=f"<b>{symbol}</b><br>Daily return: %{{x:.3f}}%<br>Count: %{{y}}<extra></extra>",
            showlegend=False,
        ),
        row=row,
        col=col,
    )
    fig.add_vline(x=0, line_dash="dot", line_color=globals().get("CHART_COLORS", {}).get("muted", "#6b7280"), row=row, col=col)

chart_title = dict(
    text="Daily Return Distributions<br><sup>Selected ETFs, shown as percent per day</sup>",
    x=0.5,
    xanchor="center",
    y=0.98,
    yanchor="top",
    font=dict(size=18),
)
fig.update_layout(
    title=chart_title,
    height=max(620, rows * 300),
    margin=dict(l=70, r=40, t=95, b=70),
    bargap=0.05,
)
fig.update_annotations(font_size=13, yshift=8)
fig.update_xaxes(title_text="Daily return (%)", automargin=True)
fig.update_yaxes(title_text="Days", automargin=True)
if "apply_theme" in globals():
    fig = apply_theme(fig, title=chart_title, height=max(660, rows * 315))
    fig.update_layout(margin=dict(l=75, r=45, t=115, b=80), bargap=0.04)
    fig.update_annotations(font_size=13, yshift=8)
    fig.update_xaxes(title_text="Daily return (%)", automargin=True)
    fig.update_yaxes(title_text="Days", automargin=True)

if "show_metric_table" in globals():
    show_metric_table(return_distribution_summary, title="Daily Return Distribution Summary", height=390).show()
    show_metric_table(near_zero_return_detail, title="Near-Zero Return Detail", height=390).show()
else:
    display(return_distribution_summary)
    display(near_zero_return_detail)

display(
    Markdown(
        "**Reading the spike near 0%:** daily ETF returns often cluster close to zero because most trading days are small moves. "
        "That does not mean every tall bar is exactly 0.00%. The near-zero table uses finer 0.10 percentage-point bands so you can see whether the concentration is exact zeros or many small positive and negative returns rounded visually toward zero."
    )
)
fig.show()

# A teaser for Notebook 02: relationships between asset returns.
if "liquidity_rankings" in globals() and "symbol" in liquidity_rankings.columns:
    correlation_candidates = [symbol for symbol in liquidity_rankings["symbol"].tolist() if symbol in returns.columns]
else:
    correlation_candidates = returns.notna().sum().sort_values(ascending=False).index.tolist()

correlation_symbols = correlation_candidates[: min(14, len(correlation_candidates))]
correlation_matrix = returns[correlation_symbols].corr().round(2)

heatmap_colorscale = [
    [0.00, globals().get("CHART_COLORS", {}).get("rose", "#be123c")],
    [0.50, globals().get("CHART_COLORS", {}).get("background", "#ffffff")],
    [1.00, globals().get("CHART_COLORS", {}).get("blue", "#2563eb")],
]

heatmap_fig = go.Figure(
    data=go.Heatmap(
        z=correlation_matrix.to_numpy(),
        x=correlation_matrix.columns,
        y=correlation_matrix.index,
        zmin=-1,
        zmax=1,
        colorscale=heatmap_colorscale,
        colorbar=dict(title="Correlation"),
        text=correlation_matrix.to_numpy(),
        texttemplate="%{text:.2f}",
        hovertemplate="<b>%{y}</b> vs <b>%{x}</b><br>Correlation: %{z:.2f}<extra></extra>",
    )
)
heatmap_title = dict(
    text="Notebook 02 Teaser:<br><sup>Which ETF Returns Moved Differently?</sup>",
    x=0.5,
    xanchor="center",
    y=0.97,
    yanchor="top",
    font=dict(size=18),
)
heatmap_fig.update_layout(
    title=heatmap_title,
    height=720,
    margin=dict(l=115, r=95, t=115, b=105),
)
heatmap_fig.update_xaxes(side="top", tickangle=-35, automargin=True, tickfont=dict(size=11))
heatmap_fig.update_yaxes(autorange="reversed", automargin=True, tickfont=dict(size=11))
if "apply_theme" in globals():
    heatmap_fig = apply_theme(heatmap_fig, title=heatmap_title, height=720)
    heatmap_fig.update_layout(margin=dict(l=115, r=95, t=115, b=105))
    heatmap_fig.update_xaxes(side="top", tickangle=-35, automargin=True, tickfont=dict(size=11))
    heatmap_fig.update_yaxes(autorange="reversed", automargin=True, tickfont=dict(size=11))
heatmap_fig.show()

pair_rows = []
for i, left_symbol in enumerate(correlation_matrix.index):
    for j, right_symbol in enumerate(correlation_matrix.columns):
        if j <= i:
            continue
        pair_rows.append(
            {
                "ETF pair": f"{left_symbol} / {right_symbol}",
                "Correlation": correlation_matrix.loc[left_symbol, right_symbol],
                "Absolute correlation": abs(correlation_matrix.loc[left_symbol, right_symbol]),
            }
        )

low_correlation_pairs = (
    pd.DataFrame(pair_rows)
    .sort_values(["Absolute correlation", "ETF pair"])
    .head(8)
    .assign(
        Correlation=lambda frame: frame["Correlation"].map(lambda value: f"{value:.2f}"),
        **{"Absolute correlation": lambda frame: frame["Absolute correlation"].map(lambda value: f"{value:.2f}")},
    )
)
correlation_teaser = low_correlation_pairs.copy()

if "show_metric_table" in globals():
    show_metric_table(correlation_teaser, title="Lowest-Correlation Pairs In This Sample", height=430).show()
else:
    display(correlation_teaser)

display(
    Markdown(
        "**Correlation teaser:** these pairs did not move as tightly together in this sample. "
        "That does not make them automatic buys, and it does not prove the future will look the same. "
        "It simply gives Notebook 02 a better question to test: can different return behavior improve a portfolio after we inspect risk, weights, and diversification?"
    )
)


def render_wrap_table(frame, title, note=None):
    header = "".join(f"<th>{escape(str(column))}</th>" for column in frame.columns)
    body_rows = []
    for row in frame.itertuples(index=False):
        cells = "".join(f"<td>{escape(str(value))}</td>" for value in row)
        body_rows.append(f"<tr>{cells}</tr>")
    note_html = f"<p class='wrap-note'>{escape(note)}</p>" if note else ""
    return HTML(
        f"""
        <style>
          .wrap-table-area {{font-family: Arial, sans-serif; line-height: 1.45; max-width: 1100px;}}
          .wrap-table-area h3 {{text-align: center; margin: 18px 0 8px;}}
          .wrap-note {{color: #374151; margin: 6px 0 14px;}}
          .wrap-scroll {{overflow-x: auto; margin: 8px 0 18px;}}
          .wrap-table {{border-collapse: collapse; width: 100%; min-width: 760px; font-size: 14px; background: #ffffff;}}
          .wrap-table th {{text-align: left; border-bottom: 2px solid #111827; padding: 9px 10px; background: #111827; color: #ffffff; font-weight: 700; white-space: nowrap;}}
          .wrap-table td {{border-bottom: 1px solid #e5e7eb; padding: 9px 10px; color: #111827; background: #ffffff; vertical-align: top;}}
          .wrap-table tbody tr:nth-child(even) td {{background: #f9fafb;}}
        </style>
        <div class="wrap-table-area">
          <h3>{escape(title)}</h3>
          {note_html}
          <div class="wrap-scroll"><table class="wrap-table"><thead><tr>{header}</tr></thead><tbody>{''.join(body_rows)}</tbody></table></div>
        </div>
        """
    )


notebook_01_artifacts = pd.DataFrame(
    [
        {
            "Object": "dataset_config",
            "Created in": "Cell 14",
            "Plain-English meaning": "The dataset location, filenames, and Colab cache path.",
            "Why it matters next": "Lets future cells reload the same teaching data clearly.",
        },
        {
            "Object": "prices, metadata, liquidity, manifest",
            "Created in": "Cell 16",
            "Plain-English meaning": "The raw dataset pieces loaded into Python.",
            "Why it matters next": "These are the source tables behind the ETF universe and return matrix.",
        },
        {
            "Object": "dataset_field_guide",
            "Created in": "Cell 17",
            "Plain-English meaning": "A mini data dictionary for the fields learners rely on most.",
            "Why it matters next": "Keeps the portfolio workflow understandable instead of mysterious.",
        },
        {
            "Object": "etf_universe",
            "Created in": "Cell 20",
            "Plain-English meaning": "One row per ETF with coverage, metadata, and liquidity context.",
            "Why it matters next": "Helps choose examples and explain why liquidity matters.",
        },
        {
            "Object": "prices_wide",
            "Created in": "Cell 23",
            "Plain-English meaning": "A date-by-symbol matrix of close prices.",
            "Why it matters next": "This is the price shape needed before calculating returns.",
        },
        {
            "Object": "returns",
            "Created in": "Cell 25",
            "Plain-English meaning": "A date-by-symbol matrix of daily linear returns.",
            "Why it matters next": "Notebook 02 starts here: risk, diversification, correlations, and portfolios all learn from returns.",
        },
        {
            "Object": "correlation_teaser",
            "Created in": "Cell 28",
            "Plain-English meaning": "A preview of ETF pairs that moved less tightly together in this sample.",
            "Why it matters next": "Sets up Notebook 02: diversification must be tested, not assumed.",
        },
        {
            "Object": "return_distribution_summary",
            "Created in": "Cell 28",
            "Plain-English meaning": "A quick look at mean, volatility, and daily tails for selected ETFs.",
            "Why it matters next": "Prepares the eye for drawdowns and risk tradeoffs.",
        },
        {
            "Object": "near_zero_return_detail",
            "Created in": "Cell 28",
            "Plain-English meaning": "A finer check of whether the tall middle histogram bars are exact zeros or small near-zero moves.",
            "Why it matters next": "Teaches students not to overread rounded chart labels.",
        },
    ]
)

notebook_01_glossary = pd.DataFrame(
    [
        {"Term": "Price", "Meaning": "A level, such as an ETF close price on one date."},
        {"Term": "Return", "Meaning": "A percentage change through time; this is what portfolio models compare."},
        {"Term": "Coverage", "Meaning": "How much usable history an ETF brings to the dataset."},
        {"Term": "Liquidity", "Meaning": "A practical trading-activity clue, not a performance forecast."},
        {"Term": "Correlation", "Meaning": "A sample relationship showing how closely two return series moved together."},
        {"Term": "Return matrix X", "Meaning": "Rows are dates, columns are assets, and values are asset returns."},
        {"Term": "Validation", "Meaning": "Checking whether a method still makes sense on data it did not use to build the portfolio."},
    ]
)

display(
    render_wrap_table(
        notebook_01_artifacts,
        title="Notebook 01 Artifact Checklist",
        note="These are the objects worth remembering before opening Notebook 02.",
    )
)
display(
    render_wrap_table(
        notebook_01_glossary,
        title="Terms You Can Now Use Confidently",
        note="This vocabulary is the bridge from data preparation to portfolio construction.",
    )
)

display(
    Markdown(
        "### What We Built In Notebook 01\n\n"
        "Notebook 01 built the foundation for portfolio research with skfolio. We set up the Colab environment, loaded a public ETF dataset, checked coverage and liquidity, reshaped prices into a clean matrix, and converted prices into the `returns` table that portfolio models can actually use. We also used normalized price charts, return distributions, and a correlation teaser to build intuition before any optimization begins.\n\n"
        "The main lesson is the workflow: validate the data, build the return matrix, inspect behavior, then let portfolio tools operate on returns instead of raw prices. If your Colab runtime resets later, the notebook file stays saved in Google Drive; rerun the setup and data cells to recreate the objects.\n\n"
        "### Preview Of Notebook 02\n\n"
        "Notebook 02 is planned to start from `returns` and move into the portfolio questions that matter: How volatile are these ETFs? Which ones move together? Which relationships actually help diversification? What does a simple baseline portfolio look like, and how does skfolio represent that portfolio object?\n\n"
        "This first notebook is completely free. If this kind of skfolio masterclass is useful, please support The Chart Truth social media channels by following, liking, commenting, or sharing. That support helps show there is real interest in continuing the series, and if there is enough interest, `skfolio Masterclass 02 - Portfolio Objects, Risk, And Diversification` will be released.\n\n"
        "### Credits And Thanks\n\n"
        "This notebook began as an internal learning project about skfolio, and I am thankful to the skfolio project and contributors for making this open-source work available. You can visit the skfolio repository here: [skfolio/skfolio](https://github.com/skfolio/skfolio).\n\n"
        "**Cell 28 complete:** Notebook 01 is ready to save, share, and use as the foundation for the next notebook in the series."
    )
)
